## === WORKFLOW ===

**Phase 1 (GEE) → `04_export_modis_igbp.ipynb`:**
- Lade MODIS MCD12Q1 IGBP für alle Jahre 2001–2024
- Reproject zu 500m EPSG:3035, Clippe auf wildE-Grid
- Export als 24-Band GeoTIFF → `MODIS_IGBP_2001_2024_500m_wildE.tif`

**Phase 2 (Lokal) → `04_export_modis_igbp.ipynb`:**
- Download via PyDrive + Validierung gegen Grid-Referenz
- *(Kein combined TIF — alle Dateien werden in Phase 3 separat geladen)*

**Phase 3 (Lokal) — diese Datei:**

| Schritt | Beschreibung |
|---|---|
| **0** | Lade alle Daten (Woody, MBA, MODIS IGBP, Ecoregions — separat) |
| **1** | Pre-Fire Landcover (MODIS IGBP Jahr vor Brand) |
| **2a–d** | LC-Trajektorien (Major-Klasse, IGBP-Einzelklasse, LC×FireCount, LC×Ecoregion) |
| **2e** | Eco-Trajektorien (1 Fire) |
| **2f** | Eco × Fire Count |
| **2g** | Fire Statistics (vektorisiert) |
| **2h** | Recovery-%-Index (loss-relativ, Fire Freq 1–4, +10yr) |
| **2i** | Heatmap Ecoregion × LC (Fläche km²/%) |
| **3** | LC-Visualisierungen (Plots 1–5) |
| **3b** | Stat. Tests LC (Kruskal-Wallis + Dunn, europa-weit & pro Ecoregion) |
| **3c** | Eco-Visualisierungen (E1–E10) |
| **3d** | Stat. Tests Ecoregion (Kruskal-Wallis + Dunn) |
| **4** | CSV-Export + Abschlussmeldung |
| **★** | Schnelle Visualisierung aus CSV (eine Zelle, ganz am Ende) |


## Kurzübersicht

| Phase | Wo | Was |
|---|---|---|
| **Phase 1** | GEE | MODIS MCD12Q1 IGBP Export (2001–2024) → 24-Band GeoTIFF |
| **Phase 2** | Lokal | Download + Validierung `MODIS_IGBP_2001_2024_500m_wildE.tif` (kein combined TIF) |
| **Phase 3** | Lokal | Analyse → Plots 1–5 (LC), E1–E10 (Eco), Stat. Tests, CSV-Export |

→ Details siehe Workflow-Zelle oben · Phase 1+2 → `04_export_modis_igbp.ipynb`


In [2]:
# === 1. IN EARTH ENGINE - MODIS MCD12Q1 IGBP (2001-2024) ===

import ee
import datetime

# Earth Engine initialisieren
try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize()


In [14]:

# === MODIS Land Cover laden ===

dataset = ee.ImageCollection('MODIS/061/MCD12Q1')

def get_modis_lc_period(year):
    """Lade MODIS IGBP Land Cover für ein bestimmtes Jahr"""
    yearly_img = dataset.filter(
        ee.Filter.calendarRange(year, year, 'year')
    ).first().select('LC_Type1').rename(f'MODIS_LC_{year}')
    
    return yearly_img

# Lade MODIS für ALLE Jahre von 2001-2024
years = list(range(2001, 2025))  # [2001, 2002, ..., 2024]

modis_lc_images = []
for year in years:
    try:
        img = get_modis_lc_period(year)
        modis_lc_images.append(img)
    except Exception as e:
        print(f"⚠️  Fehler beim Laden von {year}: {str(e)}")

print(f"✓ {len(modis_lc_images)} MODIS IGBP Jahre geladen (2001-2024)")


✓ 24 MODIS IGBP Jahre geladen (2001-2024)


In [15]:

# === 2. CLIPPE AUF WILDE-GRID & REPROJECT ZU 500M ===

grid_ee = ee.FeatureCollection("projects/ee-butzerfe/assets/wilde_only_grid_glance_eu_150km")
region = grid_ee.geometry()

# Kombiniere alle Jahre und reproject zu 500m
modis_lc_clipped = ee.Image.cat([
    img.reproject(
        crs='EPSG:3035',
        scale=500
    ).clip(region) 
    for img in modis_lc_images
])

print("✓ Auf wildE-Grid geclippt und zu 500m reproject")
print(f"✓ Gesamte Bänder im Export: {len(years)}")


✓ Auf wildE-Grid geclippt und zu 500m reproject
✓ Gesamte Bänder im Export: 24


In [16]:

# === 3. EXPORT ===

task = ee.batch.Export.image.toDrive(
    image=modis_lc_clipped,
    description='MODIS_IGBP_2001_2024_500m_wildE',
    folder=f'MODIS_GEE_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}',
    fileNamePrefix='MODIS_IGBP_2001_2024_500m_wildE',
    region=region,
    crs='EPSG:3035',
    scale=500,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)

task.start()
print("✓✓✓ MODIS IGBP 24-Jahres-Export gestartet!")
print(f"Task ID: {task.id}")
print(f"Jahre im Export: 2001-2024 (24 Bänder)")

✓✓✓ MODIS IGBP 24-Jahres-Export gestartet!
Task ID: JIHLXHGUSS3RQ7MPOBCGKDV7
Jahre im Export: 2001-2024 (24 Bänder)


In [17]:
# === DOWNLOAD MODIS IGBP EXPORT VON GOOGLE DRIVE ===

import os
from tqdm import tqdm
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

# === AUTHENTIFIZIERUNG ===

print("Starte Google Drive Authentifizierung...")

client_secrets_path = "L:/_HiWi/Ruben/wildE/client_secrets_ruben_geopy.json"

if not os.path.exists(client_secrets_path):
    print(f"⚠️  Warnung: Secrets nicht gefunden unter {client_secrets_path}")
    gauth = GoogleAuth()
else:
    print(f"✓ Secrets gefunden: {client_secrets_path}")
    GoogleAuth.DEFAULT_SETTINGS['client_config_file'] = client_secrets_path
    gauth = GoogleAuth()

try:
    gauth.LocalWebserverAuth()
    drive = GoogleDrive(gauth)
    print("✓ Google Drive Authentifizierung erfolgreich!")
except Exception as e:
    print(f"✗ Authentifizierungsfehler: {str(e)}")
    raise

# === ZIELORDNER DEFINIEREN ===

dl_folder = os.path.join(
    workDir,
    r"_Runs\05_Landcover_integrated\MODIS_GEE_tiles"
)

os.makedirs(dl_folder, exist_ok=True)
print(f"✓ Download-Ordner: {dl_folder}")

# === SUCHE NACH MODIS EXPORT-ORDNERN ===

print("\nSuche nach MODIS_GEE_* Export-Ordnern auf Google Drive...")

folder_query = "mimeType='application/vnd.google-apps.folder' and trashed=false"
all_folders = drive.ListFile({'q': folder_query}).GetList()

# Filtere nach MODIS_GEE_
folder_list = [f for f in all_folders if 'MODIS_GEE_' in f['title']]

if len(folder_list) == 0:
    print("✗ Keine MODIS_GEE_* Ordner gefunden!")
    print("Überprüfe, ob der Export abgeschlossen ist.")
else:
    print(f"✓ {len(folder_list)} MODIS_GEE_* Ordner gefunden:")
    for i, f in enumerate(folder_list, 1):
        print(f"  {i}. {f['title']} (ID: {f['id']})")
    
    # Wähle neuesten Ordner
    print("\nWähle neuesten Ordner...")
    folder_list_sorted = sorted(folder_list, key=lambda x: x['title'], reverse=True)
    selected_folder = folder_list_sorted[0]
    print(f"✓ Neuester Ordner: {selected_folder['title']}")
    
    gee_folder_id = selected_folder['id']
    export_folder_name = selected_folder['title']
    
    # === LADE DATEIEN HERUNTER ===
    
    print(f"\nLade Dateien aus '{export_folder_name}' herunter...")
    
    drive_list = drive.ListFile({'q': f"'{gee_folder_id}' in parents and trashed=false"}).GetList()
    print(f"Anzahl Dateien im Ordner: {len(drive_list)}")
    
    if len(drive_list) == 0:
        print("⚠️  Keine Dateien im Ordner gefunden!")
    else:
        downloaded_count = 0
        failed_count = 0
        failed_files = []
        
        print("\nDownloade Dateien...")
        for file in tqdm(drive_list, desc="Download"):
            try:
                file_id = drive.CreateFile({'id': file['id']})
                fname = file["title"]
                outName = os.path.join(dl_folder, fname)
                
                # Download
                file_id.GetContentFile(outName)
                
                # Aus GDrive löschen
                file_id.Delete()
                downloaded_count += 1
                
            except Exception as e:
                print(f"\n✗ Fehler beim Download von {fname}: {str(e)}")
                failed_files.append(fname)
                failed_count += 1
        
        # Lösche leeren GEE-Ordner
        try:
            folder_obj = drive.CreateFile({'id': gee_folder_id})
            folder_obj.Delete()
            print(f"\n✓ Ordner '{export_folder_name}' aus Google Drive gelöscht")
        except Exception as e:
            print(f"\n⚠️  Ordner konnte nicht gelöscht werden: {str(e)}")
        
        # === ABSCHLUSSMELDUNG ===
        print("\n" + "="*70)
        print("✓✓✓ MODIS IGBP DOWNLOAD ERFOLGREICH ABGESCHLOSSEN! ✓✓✓")
        print("="*70)
        print(f"  Anzahl heruntergeladener Dateien: {downloaded_count}")
        if failed_count > 0:
            print(f"  Fehlgeschlagene Downloads: {failed_count}")
            for fname in failed_files:
                print(f"    - {fname}")
        print(f"  Download-Ordner: {dl_folder}")
        print("="*70)
        
        # Liste heruntergeladene Dateien auf
        print(f"\nHeruntergeladene Dateien:")
        downloaded_files = os.listdir(dl_folder)
        for fname in sorted(downloaded_files):
            fpath = os.path.join(dl_folder, fname)
            size_mb = os.path.getsize(fpath) / (1024**2)
            print(f"  ✓ {fname} ({size_mb:.2f} MB)")

Starte Google Drive Authentifizierung...
✓ Secrets gefunden: L:/_HiWi/Ruben/wildE/client_secrets_ruben_geopy.json
Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=579762042005-bktd78digk65s9sht4hho28rq5s30apo.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=offline&response_type=code

Authentication successful.
✓ Google Drive Authentifizierung erfolgreich!
✓ Download-Ordner: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_GEE_tiles

Suche nach MODIS_GEE_* Export-Ordnern auf Google Drive...
✗ Keine MODIS_GEE_* Ordner gefunden!
Überprüfe, ob der Export abgeschlossen ist.


## Phase 2: Lokale Aggregation & Kombination


In [18]:
import rasterio
import numpy as np
import os
import time
from tqdm import tqdm

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

# === SCHRITT 1: Lade MODIS LC Daten ===
print("\nLade MODIS IGBP 500m Daten (2001-2024) von GEE...")
start = time.time()

modis_gee_path = os.path.join(
    workDir,
    r"_Runs\05_Landcover_integrated\MODIS_GEE_tiles\MODIS_IGBP_2001_2024_500m_wildE.tif"
)

with rasterio.open(modis_gee_path) as modis_src:
    modis_lc = modis_src.read()
    modis_meta = modis_src.meta.copy()
    modis_bands = modis_lc.shape[0]
    modis_crs = modis_src.crs
    modis_transform = modis_src.transform

print(f"✓ MODIS IGBP geladen. Dauer: {time.time() - start:.2f} Sekunden")
print(f"  Shape: {modis_lc.shape}")
print(f"  CRS: {modis_crs}")
print(f"  Bänder: {modis_bands} (2001-2024)")



Lade MODIS IGBP 500m Daten (2001-2024) von GEE...
✓ MODIS IGBP geladen. Dauer: 11.50 Sekunden
  Shape: (24, 9660, 10483)
  CRS: EPSG:3035
  Bänder: 24 (2001-2024)


In [19]:

# === SCHRITT 2: Lade Woody-Fire-MBA Datensatz ===
print("\nLade Woody-Fire-MBA kombinierter Datensatz...")
start = time.time()

woody_burned_mba_path = os.path.join(
    workDir,
    r"_Runs\03_Woody_Fire_combined\woody_burned_MBA_full_combined.tif"
)

with rasterio.open(woody_burned_mba_path) as wf_src:
    woody_fire_mba = wf_src.read()
    wf_meta = wf_src.meta.copy()
    wf_bands = woody_fire_mba.shape[0]
    wf_crs = wf_src.crs
    wf_transform = wf_src.transform
    wf_shape = (wf_src.height, wf_src.width)

print(f"✓ Woody-Fire-MBA geladen. Dauer: {time.time() - start:.2f} Sekunden")
print(f"  Shape: {woody_fire_mba.shape}")
print(f"  CRS: {wf_crs}")
print(f"  Bänder: {wf_bands}")



Lade Woody-Fire-MBA kombinierter Datensatz...
✓ Woody-Fire-MBA geladen. Dauer: 355.06 Sekunden
  Shape: (352, 9660, 10483)
  CRS: EPSG:3035
  Bänder: 352


In [20]:
# === SCHRITT 3: Validiere CRS & Geometrie ===

print("\nValidiere CRS und Raster-Geometrie...")

crs_match = modis_crs == wf_crs
shape_match = modis_lc.shape[1:] == wf_shape

print(f"  MODIS CRS: {modis_crs}")
print(f"  Woody-Fire-MBA CRS: {wf_crs}")
print(f"  CRS Match: {'✓ JA' if crs_match else '✗ NEIN'}")

print(f"\n  MODIS Shape: {modis_lc.shape[1:]}")
print(f"  Woody-Fire-MBA Shape: {wf_shape}")
print(f"  Shape Match: {'✓ JA' if shape_match else '✗ NEIN'}")

if not crs_match:
    print(f"\n⚠️  WARNUNG: CRS unterschiedlich!")
    print(f"  MODIS: {modis_crs}")
    print(f"  Woody-Fire-MBA: {wf_crs}")
    print("  → Verwende CRS des Woody-Fire-MBA Datensatzes für Output")
    
if not shape_match:
    print(f"\n⚠️  WARNUNG: Raster-Auflösung unterschiedlich!")
    print(f"  MODIS: {modis_lc.shape[1:]}")
    print(f"  Woody-Fire-MBA: {wf_shape}")
    print("  → Resample MODIS zu Woody-Fire-MBA Größe...")
    
    # Resample MODIS zu Woody-Fire-MBA Größe falls nötig
    if modis_lc.shape[1:] != wf_shape:
        from rasterio.vrt import WarpedVRT
        
        print(f"\n  Starte Resampling-Prozess...")
        print(f"    Quell-Shape: {modis_lc.shape[1:]}")
        print(f"    Ziel-Shape: {wf_shape}")
        print(f"    Resampling-Methode: Nearest Neighbor")
        
        resample_start = time.time()
        
        with WarpedVRT(
            modis_gee_path,
            crs=wf_crs,
            transform=wf_transform,
            width=wf_shape[1],
            height=wf_shape[0],
            resampling=rasterio.enums.Resampling.nearest
        ) as vrt:
            modis_lc = vrt.read()
        
        resample_duration = time.time() - resample_start
        print(f"  ✓ Resampling abgeschlossen. Dauer: {resample_duration:.2f} Sekunden")
        print(f"    Neue Shape: {modis_lc.shape}")
else:
    print(f"\n✓ Geometrie und CRS passen perfekt zusammen!")
    print(f"  Kein Resampling erforderlich.")

# === Finale Validierung ===

print(f"\n{'='*70}")
print("FINALE VALIDIERUNG VOR KOMBINATION")
print(f"{'='*70}")

# Nach Resampling nochmal prüfen
final_shape_match = modis_lc.shape[1:] == wf_shape
final_band_count_modis = modis_lc.shape[0]
final_band_count_wf = woody_fire_mba.shape[0]

print(f"✓ MODIS finale Shape: {modis_lc.shape}")
print(f"✓ Woody-Fire-MBA Shape: {woody_fire_mba.shape}")
print(f"✓ Shape Match nach Validierung: {'✓ JA' if final_shape_match else '✗ NEIN'}")
print(f"✓ MODIS Bänder: {final_band_count_modis}")
print(f"✓ Woody-Fire-MBA Bänder: {final_band_count_wf}")
print(f"✓ Gesamte Bänder nach Kombination: {final_band_count_modis + final_band_count_wf}")

if final_shape_match:
    print(f"\n✓✓ VALIDIERUNG ERFOLGREICH!")
    print(f"   Datensätze sind ready für Kombination.")
else:
    print(f"\n✗✗ VALIDIERUNG FEHLGESCHLAGEN!")
    print(f"   Shape-Mismatch nach Resampling. Abbruch.")
    raise ValueError(f"Shape mismatch: MODIS {modis_lc.shape[1:]} vs WF {wf_shape}")

print(f"{'='*70}")


Validiere CRS und Raster-Geometrie...
  MODIS CRS: EPSG:3035
  Woody-Fire-MBA CRS: EPSG:3035
  CRS Match: ✓ JA

  MODIS Shape: (9660, 10483)
  Woody-Fire-MBA Shape: (9660, 10483)
  Shape Match: ✓ JA

✓ Geometrie und CRS passen perfekt zusammen!
  Kein Resampling erforderlich.

FINALE VALIDIERUNG VOR KOMBINATION
✓ MODIS finale Shape: (24, 9660, 10483)
✓ Woody-Fire-MBA Shape: (352, 9660, 10483)
✓ Shape Match nach Validierung: ✓ JA
✓ MODIS Bänder: 24
✓ Woody-Fire-MBA Bänder: 352
✓ Gesamte Bänder nach Kombination: 376

✓✓ VALIDIERUNG ERFOLGREICH!
   Datensätze sind ready für Kombination.


In [21]:
# === SCHRITT 4: Kombiniere Datensätze ===
print("\nKombiniere Datensätze...")
combine_start = time.time()

combined = np.concatenate([woody_fire_mba, modis_lc], axis=0)
total_bands = combined.shape[0]

combine_duration = time.time() - combine_start

# Berechne Speichergröße des kombinierten Arrays
combined_size_gb = combined.nbytes / (1024**3)

print(f"✓ Kombinierter Datensatz erstellt. Dauer: {combine_duration:.2f} Sekunden")
print(f"  Woody-Fire-MBA: {wf_bands} Bänder")
print(f"  MODIS IGBP (2001-2024): {modis_bands} Bänder")
print(f"  Gesamt: {total_bands} Bänder")
print(f"  Shape: {combined.shape}")
print(f"  Speichergröße (RAM): {combined_size_gb:.2f} GB")
print(f"  Durchsatz: {combined_size_gb / combine_duration:.2f} GB/s")


Kombiniere Datensätze...
✓ Kombinierter Datensatz erstellt. Dauer: 19.55 Sekunden
  Woody-Fire-MBA: 352 Bänder
  MODIS IGBP (2001-2024): 24 Bänder
  Gesamt: 376 Bänder
  Shape: (376, 9660, 10483)
  Speichergröße (RAM): 35.46 GB
  Durchsatz: 1.81 GB/s


In [22]:

# === SCHRITT 5: Aktualisiere Metadaten ===
wf_meta.update({
    'count': total_bands,
    'compress': 'lzw',
    'tiled': True,
    'blockxsize': 512,
    'blockysize': 512,
    'BIGTIFF': 'YES'
})


In [23]:

# === SCHRITT 6: Speichere kombiniertes Raster ===
out_dir = os.path.join(workDir, "_Runs", "05_Landcover_integrated")
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "woody_burned_MBA_MODIS_IGBP_combined.tif")

print(f"\nSpeichere kombiniertes Raster nach {out_path}...")
print("(Dies kann einige Minuten dauern...)")
save_start = time.time()

with rasterio.open(out_path, "w", **wf_meta) as dst:
    dst.write(combined)
    
save_duration = time.time() - save_start
print(f"✓ Kombinierter Raster gespeichert. Dauer: {save_duration:.2f} Sekunden")



Speichere kombiniertes Raster nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\woody_burned_MBA_MODIS_IGBP_combined.tif...
(Dies kann einige Minuten dauern...)
✓ Kombinierter Raster gespeichert. Dauer: 916.48 Sekunden


In [24]:
# === SCHRITT 7: Setze Bandbeschreibungen ===
print("\nSetze Bandbeschreibungen...")

years_woody = list(range(1985, 2025))  # 41 Jahre
years_burned = list(range(2000, 2026))  # 26 Jahre
band_structure = [
    "burned", "count", "doy_1", "doy_2", "doy_3", "doy_4",
    "doy_min", "doy_max", "doy_span", "month_first", "month_last", "season_length"
]
modis_years = list(range(2001, 2025))  # [2001, 2002, ..., 2024]

# === Validiere Band-Struktur ===
print("\nValidiere Band-Struktur...")
woody_band_count = len(years_woody)
mba_band_count = len(years_burned) * len(band_structure)
modis_band_count = len(modis_years)
expected_total = woody_band_count + mba_band_count + modis_band_count

print(f"  Woody Cover Bänder: {woody_band_count} (1985-2024)")
print(f"  MBA Metriken Bänder: {mba_band_count} ({len(years_burned)} Jahre × {len(band_structure)} Metriken)")
print(f"  MODIS IGBP Bänder: {modis_band_count} (2001-2024)")
print(f"  Erwartet gesamt: {expected_total}")
print(f"  Tatsächlich vorhanden: {total_bands}")

if expected_total != total_bands:
    print(f"\n⚠️  WARNUNG: Band-Mismatch!")
    print(f"  Erwartet: {expected_total}, Tatsächlich: {total_bands}")
    print(f"  Differenz: {expected_total - total_bands}")
    print(f"  → Passe Band-Beschreibungen an tatsächliche Bandanzahl an...")
    
    # Berechne die tatsächliche Anzahl der MBA-Bänder
    actual_mba_bands = total_bands - woody_band_count - modis_band_count
    print(f"  Tatsächliche MBA Bänder: {actual_mba_bands}")

# === Setze Bandbeschreibungen ===
try:
    with rasterio.open(out_path, "r+") as dst:
        band_idx = 1
        band_descriptions_set = 0
        
        # Woody Cover Bänder (1-41)
        print(f"\n  Setze Woody Cover Bänder (1-{woody_band_count})...")
        for year in years_woody:
            if band_idx <= total_bands:
                dst.set_band_description(band_idx, f"Woody_Cover_{year}")
                band_descriptions_set += 1
            band_idx += 1
        
        # MBA Metriken Bänder (42-353)
        print(f"  Setze MBA Metriken Bänder ({woody_band_count + 1}-{woody_band_count + mba_band_count})...")
        for year in years_burned:
            for metric in band_structure:
                if band_idx <= total_bands:
                    dst.set_band_description(band_idx, f"MBA_{year}_{metric}")
                    band_descriptions_set += 1
                band_idx += 1
        
        # MODIS IGBP Bänder (354-377)
        print(f"  Setze MODIS IGBP Bänder ({woody_band_count + mba_band_count + 1}-{total_bands})...")
        for year in modis_years:
            if band_idx <= total_bands:
                dst.set_band_description(band_idx, f"MODIS_IGBP_{year}")
                band_descriptions_set += 1
            band_idx += 1
    
    print(f"\n✓ Bandbeschreibungen gesetzt: {band_descriptions_set}/{total_bands} Bänder")
    
except Exception as e:
    print(f"\n✗ Fehler beim Setzen der Bandbeschreibungen: {str(e)}")
    print(f"  Band-Index erreichte: {band_idx}")
    print(f"  Maximal erlaubte Bänder: {total_bands}")
    raise

# === ABSCHLUSSMELDUNG ===
file_size_gb = os.path.getsize(out_path) / (1024**3)

print("\n" + "="*70)
print(f"✓✓✓ KOMBINIERTER DATENSATZ MIT MODIS IGBP ERFOLGREICH ERSTELLT! ✓✓✓")
print("="*70)
print(f"  Ausgabedatei: {out_path}")
print(f"  Gesamtgröße: {file_size_gb:.2f} GB")
print(f"  Anzahl Bänder: {total_bands}")
print(f"    - Woody Cover (1985-2025): {woody_band_count} Bänder")
print(f"    - MBA Metriken (2000-2025): {mba_band_count} Bänder")
print(f"    - MODIS IGBP (2001-2024): {modis_band_count} Bänder")
print(f"  Rasterauflösung: {wf_meta['height']} × {wf_meta['width']} Pixel")
print(f"  Datentyp: {wf_meta['dtype']}")
print(f"  CRS: {wf_crs}")
print("="*70)


Setze Bandbeschreibungen...

Validiere Band-Struktur...
  Woody Cover Bänder: 40 (1985-2024)
  MBA Metriken Bänder: 312 (26 Jahre × 12 Metriken)
  MODIS IGBP Bänder: 24 (2001-2024)
  Erwartet gesamt: 376
  Tatsächlich vorhanden: 376

  Setze Woody Cover Bänder (1-40)...
  Setze MBA Metriken Bänder (41-352)...
  Setze MODIS IGBP Bänder (353-376)...

✓ Bandbeschreibungen gesetzt: 376/376 Bänder

✓✓✓ KOMBINIERTER DATENSATZ MIT MODIS IGBP ERFOLGREICH ERSTELLT! ✓✓✓
  Ausgabedatei: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\woody_burned_MBA_MODIS_IGBP_combined.tif
  Gesamtgröße: 1.64 GB
  Anzahl Bänder: 376
    - Woody Cover (1985-2025): 40 Bänder
    - MBA Metriken (2000-2025): 312 Bänder
    - MODIS IGBP (2001-2024): 24 Bänder
  Rasterauflösung: 9660 × 10483 Pixel
  Datentyp: uint8
  CRS: EPSG:3035


## Phase 3: MODIS IGBP Land Cover Analysis


In [ ]:
# === PHASE 3: LANDCOVER-STRATIFIZIERTE TRAJEKTORIEN-ANALYSE ===
# Berechnet Pre-/Post-Fire Woody Cover Trajektorien stratifiziert nach MODIS IGBP Landcover

import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

# === KONFIGURATION ===
years_woody  = list(range(1985, 2025))   # 40 Bänder (1985-2024)
years_burned = list(range(2000, 2026))   # 26 Jahre  (2000-2025)
band_structure = [
    "burned", "count", "doy_1", "doy_2", "doy_3", "doy_4",
    "doy_min", "doy_max", "doy_span", "month_first", "month_last", "season_length"
]
modis_years = list(range(2001, 2025))    # 24 Bänder (2001-2024)

# === Separate Dateipfade (kein combined TIF) ===
woody_path  = os.path.join(workDir, r"_Runs\02_Woody_Cover\woody_cover_500m_3035.tif")
burned_path = os.path.join(workDir, r"_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m_mosaic.tif")
modis_path  = os.path.join(workDir, r"_Runs\05_Landcover_integrated\MODIS_IGBP_2001_2024_500m_wildE.tif")

ecoregion_raster_path = os.path.join(workDir, r"_Runs\04_Ecoregions_MBA\ecoregions_500m_3035_MBA.tif")
ecoregion_attr_path   = os.path.join(workDir, r"_Runs\04_Ecoregions_MBA\ecoregions_attributes_with_colors.csv")

print("=" * 70)
print("LANDCOVER-STRATIFIZIERTE TRAJEKTORIEN-ANALYSE")
print("=" * 70)
print(f"  Woody Cover : {years_woody[0]}-{years_woody[-1]}  ({len(years_woody)} Bänder)")
print(f"  MBA Burned  : {years_burned[0]}-{years_burned[-1]}  ({len(years_burned)} Jahre × {len(band_structure)} Metriken)")
print(f"  MODIS IGBP  : {modis_years[0]}-{modis_years[-1]}  ({len(modis_years)} Bänder)")


LANDCOVER-STRATIFIZIERTE TRAJEKTORIEN-ANALYSE
  Woody Cover: 1985-2024 (40 Bänder)
  MBA Burned:  2000-2025 (26 Jahre)
  MODIS IGBP:  2001-2024 (24 Bänder)
  Band-Offsets: Woody=1, MBA=41, MODIS=353


In [ ]:
# === SCHRITT 0: LADE ALLE DATEN ===

print("\n0. LADE DATEN")
print("-" * 70)

# --- Grid-Eigenschaften aus woody_path (Referenz-Raster) ---
with rasterio.open(woody_path) as src:
    nodata    = src.nodata
    transform = src.transform
    crs       = src.crs
    height    = src.height
    width     = src.width

# --- Lade Woody Cover (alle 40 Jahre, 1985–2024) ---
print("Lade Woody Cover (1985-2024)...")
with rasterio.open(woody_path) as src:
    woody = src.read()   # Shape: (40, H, W)

print(f"  ✓ Woody Cover: {woody.shape}")

# --- Lade Burned Area: "burned"-Band pro Jahr (2000-2025) ---
print("Lade Burned Area (2000-2025)...")
with rasterio.open(burned_path) as src:
    burned_band_indices = [1 + i * len(band_structure) for i in range(len(years_burned))]
    burned = src.read(burned_band_indices)   # Shape: (26, H, W)

print(f"  ✓ Burned Area: {burned.shape}")

# --- Lade MODIS IGBP (alle 24 Jahre, 2001–2024) ---
print("Lade MODIS IGBP (2001-2024)...")
with rasterio.open(modis_path) as src:
    modis_igbp = src.read()   # Shape: (24, H, W)

print(f"  ✓ MODIS IGBP: {modis_igbp.shape}")

# --- Lade Ecoregion-Raster & Attribute ---
print("Lade Ecoregion-Raster...")
with rasterio.open(ecoregion_raster_path) as src:
    eco_raster = src.read(1)

ecoregion_df = pd.read_csv(ecoregion_attr_path)

eco_mapping = {}
for _, row in ecoregion_df.iterrows():
    eco_mapping[row['NUMERIC_ID']] = {
        'code':      row['code'],
        'name':      row['name'],
        'hex_color': row['hex_color']
    }

unique_eco_ids = np.unique(eco_raster[eco_raster > 0]).astype(int)
print(f"  ✓ Ecoregions: {len(unique_eco_ids)} Regionen")

print(f"\n✓ Alle Daten geladen!")



0. LADE DATEN
----------------------------------------------------------------------
Lade Woody Cover (1985-2024)...
  ✓ Woody Cover: (40, 9660, 10483)
Lade Burned Area (2000-2025)...
  ✓ Burned Area: (26, 9660, 10483)
Lade MODIS IGBP (2001-2024)...
  ✓ MODIS IGBP: (24, 9660, 10483)
Lade Ecoregion-Raster...
  ✓ Ecoregions: 11 Regionen

✓ Alle Daten geladen!


In [27]:
# === SCHRITT 1: BESTIMME PRE-FIRE LANDCOVER PRO PIXEL ===
# Für jedes Pixel das MODIS IGBP Landcover VOR dem ersten Brand verwenden

print("\n1. BESTIMME PRE-FIRE LANDCOVER PRO PIXEL")
print("-" * 70)

# IGBP Klassen-Definition
igbp_classes = {
    1: "Evergreen Needleleaf Forests",
    2: "Evergreen Broadleaf Forests",
    3: "Deciduous Needleleaf Forests",
    4: "Deciduous Broadleaf Forests",
    5: "Mixed Forests",
    6: "Closed Shrublands",
    7: "Open Shrublands",
    8: "Woody Savannas",
    9: "Savannas",
    10: "Grasslands",
    11: "Permanent Wetlands",
    12: "Cropland",
    13: "Urban and Built-up Lands",
    14: "Cropland/Natural Vegetation Mosaics",
    15: "Permanent Snow and Ice",
    16: "Barren",
    17: "Water Bodies"
}

# Lookup-Array für Major-Klassen (Index = IGBP Code, Wert = Major-Klassen-ID)
MAJOR_LC_NAMES = ['', 'Forests', 'Shrublands', 'Savannas', 'Grasslands', 
                  'Wetlands', 'Croplands', 'Urban', 'Barren/Ice', 'Water', 'Other']
igbp_to_major_id = np.array([
    0,   # 0: NoData
    1,   # 1: Evergreen Needleleaf → Forests
    1,   # 2: Evergreen Broadleaf → Forests
    1,   # 3: Deciduous Needleleaf → Forests
    1,   # 4: Deciduous Broadleaf → Forests
    1,   # 5: Mixed Forests → Forests
    2,   # 6: Closed Shrublands → Shrublands
    2,   # 7: Open Shrublands → Shrublands
    3,   # 8: Woody Savannas → Savannas
    3,   # 9: Savannas → Savannas
    4,   # 10: Grasslands
    5,   # 11: Wetlands
    6,   # 12: Cropland → Croplands
    7,   # 13: Urban
    6,   # 14: Cropland/Veg Mosaic → Croplands
    8,   # 15: Snow/Ice → Barren/Ice
    8,   # 16: Barren → Barren/Ice
    9,   # 17: Water
], dtype=np.uint8)

# --- Berechne Anzahl Brände pro Pixel ---
fire_counts = np.sum(burned == 1, axis=0)
has_fire = fire_counts > 0

# --- Bestimme erstes Brandjahr-Index pro Pixel (vektorisiert) ---
first_fire_idx = np.argmax(burned == 1, axis=0)  # Index in years_burned

# --- Berechne MODIS-Index für Pre-Fire LC (vektorisiert) ---
print("Berechne Pre-Fire Landcover Zuordnung (vektorisiert)...")

# Brandjahr pro Pixel → MODIS-Ziel-Jahr = Brandjahr - 1
fire_year_values = np.array(years_burned)[first_fire_idx]  # shape: (H, W)
target_years = fire_year_values - 1

# Clippe auf MODIS-Verfügbarkeit [2001, 2024]
modis_idx = np.clip(target_years - modis_years[0], 0, len(modis_years) - 1)

# Extrahiere MODIS LC per Fancy Indexing
rows, cols = np.where(has_fire)
pre_fire_lc = np.zeros((height, width), dtype=np.uint8)
pre_fire_lc[rows, cols] = modis_igbp[modis_idx[rows, cols], rows, cols]

# Filtere ungültige Werte (nur 1-17 behalten)
invalid = (pre_fire_lc < 1) | (pre_fire_lc > 17)
pre_fire_lc[invalid] = 0

# --- Erstelle Major-Klassen-ID Raster (vektorisiert via Lookup) ---
pre_fire_lc_major_id = igbp_to_major_id[pre_fire_lc]  # shape: (H, W), dtype: uint8

# Statistik
unique_lc, lc_counts = np.unique(pre_fire_lc[pre_fire_lc > 0], return_counts=True)
print(f"\n✓ Pre-Fire Landcover zugeordnet für {np.sum(pre_fire_lc > 0):,} gebrannte Pixel")
print(f"\nVerteilung der Pre-Fire IGBP Klassen:")
for lc, cnt in zip(unique_lc, lc_counts):
    pct = cnt / lc_counts.sum() * 100
    print(f"  {lc:2d} ({igbp_classes.get(lc, '?'):40s}): {cnt:>8,} Pixel ({pct:.1f}%)")

# Major-Klassen Statistik
unique_major, major_counts = np.unique(pre_fire_lc_major_id[pre_fire_lc_major_id > 0], return_counts=True)
print(f"\nVerteilung der Major Landcover-Klassen:")
for mid, cnt in zip(unique_major, major_counts):
    pct = cnt / major_counts.sum() * 100
    print(f"  {MAJOR_LC_NAMES[mid]:20s}: {cnt:>8,} Pixel ({pct:.1f}%)")


1. BESTIMME PRE-FIRE LANDCOVER PRO PIXEL
----------------------------------------------------------------------
Berechne Pre-Fire Landcover Zuordnung (vektorisiert)...

✓ Pre-Fire Landcover zugeordnet für 4,301,671 gebrannte Pixel

Verteilung der Pre-Fire IGBP Klassen:
   1 (Evergreen Needleleaf Forests            ):   45,746 Pixel (1.1%)
   2 (Evergreen Broadleaf Forests             ):    6,240 Pixel (0.1%)
   3 (Deciduous Needleleaf Forests            ):       94 Pixel (0.0%)
   4 (Deciduous Broadleaf Forests             ):   55,104 Pixel (1.3%)
   5 (Mixed Forests                           ):   74,072 Pixel (1.7%)
   6 (Closed Shrublands                       ):    1,647 Pixel (0.0%)
   7 (Open Shrublands                         ):   14,483 Pixel (0.3%)
   8 (Woody Savannas                          ):  202,451 Pixel (4.7%)
   9 (Savannas                                ):  246,497 Pixel (5.7%)
  10 (Grasslands                              ): 1,064,279 Pixel (24.7%)
  11 (Permanent W

In [29]:
# === SCHRITT 2: BERECHNE TRAJEKTORIEN NACH LANDCOVER ===

print("\n2. BERECHNE TRAJEKTORIEN NACH LANDCOVER")
print("-" * 70)

def calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, 
                    rel_years=range(-5, 11)):
    """
    Berechnet Pre-/Post-Fire Woody Cover Trajektorie für maskierte Pixel.
    Verwendet NUR Pixel mit GENAU 1 Brandereignis.
    """
    single_fire_mask = (fire_counts == 1) & mask
    
    n_pixels = np.sum(single_fire_mask)
    
    if n_pixels < 30:
        return None
    
    offset = years_burned[0] - years_woody[0]  # 2000 - 1985 = 15
    
    # Vorberechne Indices nur für relevante Pixel
    y_all, x_all = np.where(single_fire_mask)
    fire_idx_subset = first_fire_idx[y_all, x_all]
    
    trajectory = []
    trajectory_std = []
    trajectory_n = []
    
    for rel_year in rel_years:
        woody_band_idx = fire_idx_subset + rel_year + offset - 1
        
        # Filtere gültige Band-Indices
        valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))
        
        if np.sum(valid) == 0:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            trajectory_n.append(0)
            continue
        
        y_valid = y_all[valid]
        x_valid = x_all[valid]
        b_valid = woody_band_idx[valid]
        
        values = woody[b_valid, y_valid, x_valid].astype(float)
        values = values[(values != nodata) & (~np.isnan(values))]
        
        if len(values) > 0:
            trajectory.append(np.nanmean(values))
            trajectory_std.append(np.nanstd(values))
            trajectory_n.append(len(values))
        else:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            trajectory_n.append(0)
    
    return {
        'trajectory': trajectory,
        'std': trajectory_std,
        'n_per_year': trajectory_n,
        'n_pixels': int(n_pixels)
    }


def calc_trajectory_by_fire_count(woody, burned, mask, years_woody, years_burned, 
                                   nodata, fire_count_value, rel_years=range(-5, 11)):
    """
    Berechnet Trajektorie für Pixel mit bestimmter Brandanzahl.
    """
    specific_mask = (fire_counts == fire_count_value) & mask
    
    n_pixels = np.sum(specific_mask)
    
    if n_pixels < 30:
        return None
    
    offset = years_burned[0] - years_woody[0]
    
    y_all, x_all = np.where(specific_mask)
    fire_idx_subset = first_fire_idx[y_all, x_all]
    
    trajectory = []
    trajectory_std = []
    
    for rel_year in rel_years:
        woody_band_idx = fire_idx_subset + rel_year + offset - 1
        
        valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))
        
        if np.sum(valid) == 0:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            continue
        
        values = woody[woody_band_idx[valid], y_all[valid], x_all[valid]].astype(float)
        values = values[(values != nodata) & (~np.isnan(values))]
        
        if len(values) > 0:
            trajectory.append(np.nanmean(values))
            trajectory_std.append(np.nanstd(values))
        else:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
    
    return {
        'trajectory': trajectory,
        'std': trajectory_std,
        'n_pixels': int(n_pixels)
    }


rel_years = list(range(-5, 11))  # -5 bis +10 Jahre relativ zum Brand

# === Major-Klassen die analysiert werden sollen ===
# IDs: 1=Forests, 2=Shrublands, 3=Savannas, 4=Grasslands, 5=Wetlands, 6=Croplands
# Skip: 7=Urban, 8=Barren/Ice, 9=Water
ANALYZE_MAJOR_IDS = [1, 2, 3, 4, 5, 6, 8]  # inkl. Barren/Ice falls gewünscht
SKIP_MAJOR_IDS = {0, 7, 9, 10}  # NoData, Urban, Water, Other

major_classes_present = sorted(set(
    int(mid) for mid in unique_major if mid not in SKIP_MAJOR_IDS
))

print(f"Analysiere {len(major_classes_present)} Major LC Klassen: "
      f"{[MAJOR_LC_NAMES[m] for m in major_classes_present]}")

# === 2a: Trajektorien nach IGBP Major-Klasse (1 Fire) ===
print("\n2a. Trajektorien nach IGBP Major-Klasse (1 Fire Event)...")

lc_major_results = {}

for major_id in tqdm(major_classes_present, desc="Major LC Klassen"):
    lc_name = MAJOR_LC_NAMES[major_id]
    
    # Vektorisierte Maske statt List-Comprehension!
    mask = (pre_fire_lc_major_id == major_id)
    
    result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
    
    if result is not None:
        lc_major_results[lc_name] = result
        print(f"  ✓ {lc_name}: {result['n_pixels']:,} Pixel")
    else:
        print(f"  ✗ {lc_name}: zu wenige Pixel (<30)")

# === 2b: Trajektorien nach IGBP Einzelklasse (1 Fire) ===
print("\n2b. Trajektorien nach IGBP Einzelklasse (1 Fire Event)...")

lc_detail_results = {}

for lc_code in tqdm(unique_lc, desc="IGBP Klassen"):
    if lc_code in [13, 15, 16, 17]:  # Skip Urban, Ice, Barren, Water
        continue
    
    # Vektorisierte Maske!
    mask = (pre_fire_lc == lc_code)
    result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
    
    if result is not None:
        lc_detail_results[int(lc_code)] = result
        print(f"  ✓ {lc_code} ({igbp_classes[lc_code]}): {result['n_pixels']:,} Pixel")

# === 2c: Trajektorien nach Landcover × Fire Count ===
print("\n2c. Trajektorien nach Landcover × Fire Count...")

lc_firecount_results = {}

for major_id in tqdm(major_classes_present, desc="LC × Fire Count"):
    lc_name = MAJOR_LC_NAMES[major_id]
    mask = (pre_fire_lc_major_id == major_id)
    
    lc_firecount_results[lc_name] = {}
    
    for fc in range(1, 5):
        result = calc_trajectory_by_fire_count(
            woody, burned, mask, years_woody, years_burned, nodata, fc, rel_years
        )
        if result is not None:
            lc_firecount_results[lc_name][fc] = result
            print(f"  ✓ {lc_name} × {fc} Fire(s): {result['n_pixels']:,} Pixel")

# === 2d: Trajektorien nach Landcover × Ecoregion (1 Fire) ===
print("\n2d. Trajektorien nach Landcover × Ecoregion (1 Fire)...")

lc_eco_results = {}

for eco_id in tqdm(unique_eco_ids, desc="LC × Ecoregion"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)  # Vorberechnet pro Ecoregion
    lc_eco_results[eco_code] = {}
    
    for major_id in major_classes_present:
        lc_name = MAJOR_LC_NAMES[major_id]
        
        # Vektorisierte kombinierte Maske!
        mask = (pre_fire_lc_major_id == major_id) & eco_mask
        
        result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
        
        if result is not None:
            lc_eco_results[eco_code][lc_name] = result

print(f"\n✓ Alle Trajektorien berechnet!")


2. BERECHNE TRAJEKTORIEN NACH LANDCOVER
----------------------------------------------------------------------
Analysiere 7 Major LC Klassen: ['Forests', 'Shrublands', 'Savannas', 'Grasslands', 'Wetlands', 'Croplands', 'Barren/Ice']

2a. Trajektorien nach IGBP Major-Klasse (1 Fire Event)...


Major LC Klassen:   0%|          | 0/7 [00:00<?, ?it/s]

Major LC Klassen:  14%|█▍        | 1/7 [00:00<00:05,  1.17it/s]

  ✓ Forests: 157,808 Pixel


Major LC Klassen:  29%|██▊       | 2/7 [00:01<00:03,  1.39it/s]

...

  ✓ Croplands × 4 Fire(s): 116,822 Pixel
  ✓ Barren/Ice × 1 Fire(s): 5,485 Pixel
  ✓ Barren/Ice × 2 Fire(s): 646 Pixel
  ✓ Barren/Ice × 3 Fire(s): 128 Pixel


LC × Fire Count: 100%|██████████| 7/7 [00:21<00:00,  3.12s/it]



2d. Trajektorien nach Landcover × Ecoregion (1 Fire)...


LC × Ecoregion: 100%|██████████| 11/11 [00:49<00:00,  4.52s/it]


✓ Alle Trajektorien berechnet!


In [30]:
# === SCHRITT 2e: TRAJEKTORIEN PRO ECOREGION (1 Fire) ===

print("\n2e. Trajektorien pro Ecoregion (1 Fire Event)...")

eco_results = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregions (1 Fire)"):
    eco_code = eco_mapping[eco_id]['code']
    eco_name = eco_mapping[eco_id]['name']
    eco_color = eco_mapping[eco_id]['hex_color']
    eco_mask = (eco_raster == eco_id)
    
    result = calc_trajectory(
        woody, burned, eco_mask, years_woody, years_burned, nodata, rel_years
    )
    
    if result is not None:
        eco_results[eco_code] = {
            **result,
            'name': eco_name,
            'hex_color': eco_color,
            'eco_id': int(eco_id)
        }
        print(f"  ✓ {eco_code} ({eco_name}): {result['n_pixels']:,} Pixel")
    else:
        print(f"  ✗ {eco_code} ({eco_name}): zu wenige Pixel (<30)")

print(f"\n✓ {len(eco_results)} Ecoregion-Trajektorien berechnet")


2e. Trajektorien pro Ecoregion (1 Fire Event)...


Ecoregions (1 Fire):   9%|▉         | 1/11 [00:00<00:07,  1.39it/s]

  ✓ Anatolian (Anatolian Bio-geographical Region): 74,792 Pixel


Ecoregions (1 Fire):  18%|█▊        | 2/11 [00:01<00:06,  1.49it/s]

  ✓ BlackSea (Black Sea Bio-geographical Region): 8,342 Pixel


...

  ✓ Atlantic (Atlantic Bio-geographical Region): 37,725 Pixel


Ecoregions (1 Fire):  91%|█████████ | 10/11 [00:09<00:00,  1.21it/s]

  ✓ Outside (outside Europe): 165,297 Pixel


Ecoregions (1 Fire): 100%|██████████| 11/11 [00:10<00:00,  1.06it/s]

  ✓ Arctic (Arctic Bio-geographical Region): 915 Pixel

✓ 11 Ecoregion-Trajektorien berechnet


In [31]:
# === SCHRITT 2f: TRAJEKTORIEN PRO ECOREGION × FIRE COUNT ===

print("\n2f. Trajektorien pro Ecoregion × Fire Count...")

eco_firecount_results = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregion × Fire Count"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    
    eco_firecount_results[eco_code] = {}
    
    for fc in range(1, 5):
        result = calc_trajectory_by_fire_count(
            woody, burned, eco_mask, years_woody, years_burned, nodata, fc, rel_years
        )
        if result is not None:
            eco_firecount_results[eco_code][fc] = result
            print(f"  ✓ {eco_code} × {fc} Fire(s): {result['n_pixels']:,} Pixel")

print(f"\n✓ Ecoregion × Fire Count Trajektorien berechnet")


2f. Trajektorien pro Ecoregion × Fire Count...


Ecoregion × Fire Count:   0%|          | 0/11 [00:00<?, ?it/s]

  ✓ Anatolian × 1 Fire(s): 74,792 Pixel
  ✓ Anatolian × 2 Fire(s): 27,563 Pixel
  ✓ Anatolian × 3 Fire(s): 14,772 Pixel


Ecoregion × Fire Count:   9%|▉         | 1/11 [00:02<00:24,  2.49s/it]

  ✓ Anatolian × 4 Fire(s): 9,544 Pixel
  ✓ BlackSea × 1 Fire(s): 8,342 Pixel
  ✓ BlackSea × 2 Fire(s): 2,987 Pixel
  ✓ BlackSea × 3 Fire(s): 1,644 Pixel


...

  ✓ Atlantic × 4 Fire(s): 454 Pixel
  ✓ Outside × 1 Fire(s): 165,297 Pixel
  ✓ Outside × 2 Fire(s): 62,638 Pixel
  ✓ Outside × 3 Fire(s): 37,407 Pixel


Ecoregion × Fire Count:  91%|█████████ | 10/11 [00:28<00:02,  2.64s/it]

  ✓ Outside × 4 Fire(s): 24,589 Pixel
  ✓ Arctic × 1 Fire(s): 915 Pixel


Ecoregion × Fire Count: 100%|██████████| 11/11 [00:30<00:00,  2.73s/it]


✓ Ecoregion × Fire Count Trajektorien berechnet


In [ ]:
# === SCHRITT 2g: FIRE STATISTICS (vektorisiert) ===
# Fire Season Length, Fire Count, Burned Area – alles ohne Python-Loops über Pixel

print("\n2g. FIRE STATISTICS (vektorisiert)")
print("-" * 70)

import time
stats_start = time.time()

# === Lade MBA season_length und count Bänder ===
print("Lade MBA Fire Count & Season Length Bänder...")

with rasterio.open(burned_path) as src:
    # "count" = Index 1 in band_structure, "season_length" = Index 11
    count_band_indices  = [1 + i * len(band_structure) + 1  for i in range(len(years_burned))]
    season_band_indices = [1 + i * len(band_structure) + 11 for i in range(len(years_burned))]

    mba_count  = src.read(count_band_indices)     # (26, H, W)
    mba_season = src.read(season_band_indices)    # (26, H, W)

print(f"  ✓ MBA Count: {mba_count.shape}")
print(f"  ✓ MBA Season Length: {mba_season.shape}")

# === Pixelfläche ===
pixel_area_km2 = 0.25   # 500m × 500m
pixel_area_ha = 25.0



2g. FIRE STATISTICS (vektorisiert)
----------------------------------------------------------------------
Lade MBA Fire Count & Season Length Bänder...
  ✓ MBA Count: (26, 9660, 10483)
  ✓ MBA Season Length: (26, 9660, 10483)

A) Jährlich verbrannte Fläche (gesamt)...
  ✓ Gesamte verbrannte Fläche: 1,945,125 km²
  ✓ Mittlere jährl. Fläche: 74,812 km²/yr
  ✓ Maximum: 143,319 km² (2008)

B) Jährlich verbrannte Fläche pro Ecoregion (vektorisiert)...


Burned Area pro Eco: 100%|██████████| 11/11 [00:11<00:00,  1.03s/it]


  ✓ 286 Zeilen (Eco × Jahr)

C) Fire Count Verteilung pro Ecoregion...
  ✓ Fire Count Verteilung: 153 Einträge

D) Fire Season Length Statistiken pro Ecoregion × Jahr...


Season Length Stats: 100%|██████████| 11/11 [00:36<00:00,  3.29s/it]


  ✓ Season Length Stats: 273 Einträge

E) Season Length abhängig von Fire Count...
  1 Fire(s): n=7,666,446, mean=1.0 months
  2 Fire(s): n=112,814, mean=4.0 months
  3 Fire(s): n=1,229, mean=7.2 months
  4 Fire(s): n=11, mean=10.4 months

F) Aggregierte Fire Statistics pro Ecoregion...

✓ Alle Fire Statistics berechnet in 214.4s


In [ ]:
# === SCHRITT 2h: RECOVERY-%-INDEX (loss-relativ, Fire Freq 1–4, bis +10yr) ===

print("\n2h. RECOVERY-%-INDEX (loss-relativ, Fire Freq 1–4, bis +10yr)")
print("-" * 70)

eco_out_dir = os.path.join(workDir, "_Runs", "04_Ecoregions_MBA", "analysis")
os.makedirs(eco_out_dir, exist_ok=True)

colors_fc = {1: '#1f77b4', 2: '#ff7f0e', 3: '#2ca02c', 4: '#d62728'}


def calc_trajectory_extended(woody, burned, eco_mask, years_woody, years_burned,
                              nodata, fire_count_value, fire_counts_arr, first_fire_idx_arr):
    """
    Berechnet Extended Trajectory für Pixel mit spezifischem Fire Count (−5 bis +10 Jahre).
    Nutzt vorberechnete fire_counts_arr und first_fire_idx_arr aus Step 1.
    """
    specific_fire_mask = (fire_counts_arr == fire_count_value) & eco_mask
    n_pixels = int(np.sum(specific_fire_mask))
    if n_pixels < 30:
        return None

    offset = years_burned[0] - years_woody[0]
    trajectory, trajectory_std = [], []

    for rel_year in range(-5, 11):
        woody_band = first_fire_idx_arr + rel_year + offset
        valid_mask = (woody_band >= 0) & (woody_band < len(years_woody)) & specific_fire_mask

        if np.sum(valid_mask) == 0:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            continue

        y_idx, x_idx = np.where(valid_mask)
        values = woody[woody_band[y_idx, x_idx], y_idx, x_idx]
        values = values[values != nodata]

        if len(values) > 0:
            trajectory.append(float(np.nanmean(values)))
            trajectory_std.append(float(np.nanstd(values)))
        else:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)

    return {'trajectory': trajectory, 'std': trajectory_std, 'n_pixels': n_pixels}


# --- Extended Trajectories berechnen ---
print("Berechne Extended Trajectories (10yr post-fire) pro Ecoregion × Fire Count...")
extended_trajectories = {}

for eco_id in tqdm(unique_eco_ids, desc="Extended Trajectories"):
    eco_code = eco_mapping[eco_id]['code']
    if eco_code not in eco_results:
        continue
    eco_mask = (eco_raster == eco_id)
    extended_trajectories[eco_code] = {}
    for fc_val in range(1, 5):
        result = calc_trajectory_extended(
            woody, burned, eco_mask, years_woody, years_burned,
            nodata, fc_val, fire_counts, first_fire_idx
        )
        if result is not None:
            extended_trajectories[eco_code][fc_val] = result

# --- Recovery-% berechnen ---
recovery_data_extended = []

for eco_code, fc_dict in extended_trajectories.items():
    for fc_val, traj_data in fc_dict.items():
        traj_arr = np.array(traj_data['trajectory'])   # 16 Werte: −5 bis +10
        pre_fire  = float(np.nanmean(traj_arr[:5]))    # Jahre −5 bis −1
        fire_year = float(traj_arr[5])                 # Jahr 0
        loss = pre_fire - fire_year

        for years_after in range(1, 11):
            rec_val = float(traj_arr[5 + years_after])
            rec_pct = ((rec_val - fire_year) / loss * 100) if loss > 0 else np.nan
            recovery_data_extended.append({
                'Ecoregion_Code':   eco_code,
                'Fire_Count':       fc_val,
                'N_Pixels':         traj_data['n_pixels'],
                'Years_After_Fire': years_after,
                'Pre_Fire_Cover':   pre_fire,
                'Fire_Year_Cover':  fire_year,
                'Recovery_Cover':   rec_val,
                'Woody_Loss':       loss,
                'Recovery_Percent': rec_pct
            })

recovery_df_extended = pd.DataFrame(recovery_data_extended)

# --- Plot: 2×2 Recovery % pro Fire Count ---
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for fc_idx, fc_val in enumerate([1, 2, 3, 4]):
    ax = axes[fc_idx // 2, fc_idx % 2]
    fc_data = recovery_df_extended[recovery_df_extended['Fire_Count'] == fc_val]
    all_vals = fc_data['Recovery_Percent'].dropna().values
    y_min = min(all_vals.min() - 10, -50) if len(all_vals) > 0 else -50
    y_max = max(all_vals.max() + 10, 120) if len(all_vals) > 0 else 120

    for eco_code in sorted(fc_data['Ecoregion_Code'].unique()):
        eco_data = fc_data[fc_data['Ecoregion_Code'] == eco_code]
        color = next(
            (eco_mapping[eid]['hex_color'] for eid in unique_eco_ids
             if eco_mapping[eid]['code'] == eco_code),
            '#808080'
        )
        ax.plot(eco_data['Years_After_Fire'], eco_data['Recovery_Percent'],
                marker='o', linewidth=2.5, markersize=8,
                label=f"{eco_code} (n={eco_data['N_Pixels'].iloc[0]:,})",
                color=color, alpha=0.85)

    ax.axhline(y=100, color='green', linestyle='--', linewidth=2, alpha=0.6, label='Full Recovery')
    ax.axhline(y=0,   color='gray',  linestyle=':',  linewidth=1.5, alpha=0.5, label='No Recovery')
    ax.set_xlabel('Years After Fire', fontsize=12)
    ax.set_ylabel('Recovery (%)', fontsize=12)
    ax.set_title(f'Recovery Rate: {fc_val} Fire Event(s)', fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.5, 10.5)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(range(1, 11))

plt.suptitle('Woody Cover Recovery Rates by Fire Count (bis +10 Jahre)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_out_dir, "recovery_rates_by_fire_count_extended.png"), dpi=300, bbox_inches='tight')
plt.close()
print("✓ Plot: recovery_rates_by_fire_count_extended.png")

recovery_df_extended.to_csv(os.path.join(eco_out_dir, "recovery_rates_by_fire_count_extended.csv"), index=False)
print("✓ CSV:  recovery_rates_by_fire_count_extended.csv")

# Kurzübersicht
print("\n" + "=" * 70)
for fc_val in [1, 2, 3, 4]:
    print(f"\n{fc_val} FIRE EVENT(S):")
    fc_data = recovery_df_extended[recovery_df_extended['Fire_Count'] == fc_val]
    for eco_code in sorted(fc_data['Ecoregion_Code'].unique()):
        eco_data = fc_data[fc_data['Ecoregion_Code'] == eco_code]
        r5  = eco_data[eco_data['Years_After_Fire'] == 5 ]['Recovery_Percent'].values
        r10 = eco_data[eco_data['Years_After_Fire'] == 10]['Recovery_Percent'].values
        r5_str  = f"{r5[0]:.1f}%"  if len(r5)  > 0 and not np.isnan(r5[0])  else "n/a"
        r10_str = f"{r10[0]:.1f}%" if len(r10) > 0 and not np.isnan(r10[0]) else "n/a"
        print(f"  {eco_code}: 5yr={r5_str}, 10yr={r10_str}")
print("=" * 70)


In [ ]:
# === SCHRITT 2i: HEATMAP Ecoregion × Fire Count (Fläche km²/%) + Ecoregion × LC ===

print("\n2i. HEATMAP ECOREGION × FIRE COUNT (Fläche km²/%) + ECOREGION × LC")
print("-" * 70)

pixel_area_km2 = 0.25   # 500m × 500m
pixel_area_ha  = 25.0

# Eco-Gesamtflächen (aus eco_raster)
eco_total_areas_km2 = {
    eco_mapping[eco_id]['code']: float(np.sum(eco_raster == eco_id)) * pixel_area_km2
    for eco_id in unique_eco_ids
}

# --- Summary Statistics: Ecoregion × Fire Count ---
summary_stats = []

for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_name = eco_mapping[eco_id]['name']
    if eco_code not in eco_firecount_results:
        continue
    eco_total_area = eco_total_areas_km2[eco_code]

    for fc_val, traj_data in eco_firecount_results[eco_code].items():
        traj_arr     = np.array(traj_data['trajectory'])   # 16 Werte: −5 bis +10
        pre_fire     = float(np.nanmean(traj_arr[:5]))     # Jahre −5 bis −1
        fire_year    = float(traj_arr[5])                  # Jahr 0
        rec_5yr      = float(traj_arr[10])                 # Jahr +5
        rec_10yr     = float(traj_arr[15]) if len(traj_arr) > 15 else np.nan  # Jahr +10
        loss         = pre_fire - fire_year
        rec_pct_5yr  = ((rec_5yr  - fire_year) / loss * 100) if loss > 0 else np.nan
        rec_pct_10yr = ((rec_10yr - fire_year) / loss * 100) if (loss > 0 and not np.isnan(rec_10yr)) else np.nan
        area_km2     = traj_data['n_pixels'] * pixel_area_km2

        summary_stats.append({
            'Ecoregion_Code':        eco_code,
            'Ecoregion_Name':        eco_name,
            'Fire_Count':            fc_val,
            'N_Pixels':              traj_data['n_pixels'],
            'Area_km2':              area_km2,
            'Area_ha':               traj_data['n_pixels'] * pixel_area_ha,
            'Percent_of_Ecoregion':  (area_km2 / eco_total_area * 100) if eco_total_area > 0 else 0,
            'Ecoregion_Total_km2':   eco_total_area,
            'Pre_Fire_Cover':        pre_fire,
            'Post_Fire_Cover':       fire_year,
            'Recovery_5yr_Cover':    rec_5yr,
            'Woody_Loss':            loss,
            'Recovery_Percent_5yr':  rec_pct_5yr,
            'Recovery_Percent_10yr': rec_pct_10yr
        })

summary_df_eco = pd.DataFrame(summary_stats)
eco_codes_summary = sorted(summary_df_eco['Ecoregion_Code'].unique())

summary_csv = os.path.join(eco_out_dir, "summary_statistics_by_fire_count.csv")
summary_df_eco.to_csv(summary_csv, index=False)
print(f"✓ CSV: {summary_csv}")

# --- Heatmap 1: Ecoregion × Fire Count (4 Metriken) ---
print("Erstelle Heatmap Ecoregion × Fire Count...")

metrics_fc  = ['Area_km2', 'Percent_of_Ecoregion', 'Woody_Loss', 'Recovery_Percent_5yr']
titles_fc   = ['Betroffene Fläche (km²)', 'Betroffene Fläche (% der Region)',
               'Woody Cover Verlust (%)', 'Recovery nach 5 Jahren (%)']
cmaps_fc    = ['YlOrRd', 'YlOrRd', 'Reds', 'RdYlGn']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for idx, (metric, title, cmap) in enumerate(zip(metrics_fc, titles_fc, cmaps_fc)):
    ax = axes[idx // 2, idx % 2]
    pivot = summary_df_eco.pivot(index='Fire_Count', columns='Ecoregion_Code', values=metric)
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap)

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticklabels([f'{int(fc_val)} Fire(s)' for fc_val in pivot.index])

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                text_color = 'white' if val > np.nanmean(pivot.values) else 'black'
                txt = f'{val:.1f}' if metric == 'Area_km2' else f'{val:.1f}%'
                ax.text(j, i, txt, ha='center', va='center',
                        color=text_color, fontsize=10, fontweight='bold')

    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Ecoregion', fontsize=11)
    ax.set_ylabel('Fire Count', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=9)

plt.suptitle('Summary Heatmap: Ecoregion × Fire Count', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(eco_out_dir, "summary_heatmap_ecoregion_firecount.png"), dpi=300, bbox_inches='tight')
plt.close()
print("✓ Heatmap: summary_heatmap_ecoregion_firecount.png")

# --- Heatmap 2: Ecoregion × Pre-Fire LC (3 Metriken) ---
print("Erstelle Heatmap Ecoregion × Pre-Fire LC...")

lc_classes_avail = sorted({lc for eco in lc_eco_results for lc in lc_eco_results[eco]})
eco_codes_lc     = sorted(lc_eco_results.keys())

metrics_lc = ['loss', 'recovery_5yr', 'n_pixels']
titles_lc  = ['Sofort-Verlust (%)', 'Recovery 5yr (%)', 'Anzahl Pixel (n)']
cmaps_lc   = ['Reds', 'RdYlGn', 'Blues']

fig, axes_lc = plt.subplots(1, 3, figsize=(22, max(6, len(eco_codes_lc) * 0.8 + 2)))

for idx, (metric, title, cmap) in enumerate(zip(metrics_lc, titles_lc, cmaps_lc)):
    ax = axes_lc[idx]
    mat = np.full((len(eco_codes_lc), len(lc_classes_avail)), np.nan)

    for i, eco_code in enumerate(eco_codes_lc):
        for j, lc_name in enumerate(lc_classes_avail):
            if lc_name not in lc_eco_results.get(eco_code, {}):
                continue
            td       = lc_eco_results[eco_code][lc_name]
            ta       = np.array(td['trajectory'])
            pre      = float(np.nanmean(ta[:5]))
            fire     = float(ta[5])
            rec      = float(ta[10])
            loss_val = pre - fire
            rec_5pct = ((rec - fire) / loss_val * 100) if loss_val > 0 else np.nan

            if metric == 'loss':
                mat[i, j] = loss_val
            elif metric == 'recovery_5yr':
                mat[i, j] = rec_5pct
            else:
                mat[i, j] = td['n_pixels']

    im = ax.imshow(np.ma.masked_invalid(mat), aspect='auto', cmap=cmap)
    ax.set_xticks(np.arange(len(lc_classes_avail)))
    ax.set_yticks(np.arange(len(eco_codes_lc)))
    ax.set_xticklabels(lc_classes_avail, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(eco_codes_lc, fontsize=9)

    valid = mat[~np.isnan(mat)]
    mean_val = valid.mean() if len(valid) > 0 else 0
    for i in range(len(eco_codes_lc)):
        for j in range(len(lc_classes_avail)):
            val = mat[i, j]
            if not np.isnan(val):
                text_color = 'white' if val > mean_val else 'black'
                txt = f'{val:.0f}' if metric == 'n_pixels' else f'{val:.1f}%'
                ax.text(j, i, txt, ha='center', va='center',
                        color=text_color, fontsize=8, fontweight='bold')

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Pre-Fire Land Cover', fontsize=10)
    ax.set_ylabel('Ecoregion', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=9)

plt.suptitle('Ecoregion × Pre-Fire Land Cover: Loss & Recovery (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_out_dir, "summary_heatmap_ecoregion_lc.png"), dpi=300, bbox_inches='tight')
plt.close()
print("✓ Heatmap: summary_heatmap_ecoregion_lc.png")

print("\n" + "=" * 70)
print("✓✓✓ SCHRITT 2i ABGESCHLOSSEN ✓✓✓")
print("=" * 70)
print(f"  CSV:    {summary_csv}")
print(f"  Plots:  summary_heatmap_ecoregion_firecount.png")
print(f"          summary_heatmap_ecoregion_lc.png")
print("=" * 70)


In [33]:
# # === SCHNELLE VISUALISIERUNG AUS CSV (ohne Neuberechnung) ===
# # Nutze diese Zelle, wenn du nur Plots anpassen willst

# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import os

# workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
# out_dir = os.path.join(workDir, "_Runs", "05_Landcover_integrated", "MODIS_IGBP_Analysis")

# # --- Lade CSVs ---
# df_traj_major = pd.read_csv(os.path.join(out_dir, "trajectories_by_landcover_1fire.csv"))
# df_traj_fc = pd.read_csv(os.path.join(out_dir, "trajectories_by_landcover_x_firecount.csv"))
# df_traj_eco = pd.read_csv(os.path.join(out_dir, "trajectories_by_landcover_x_ecoregion.csv"))
# df_recovery = pd.read_csv(os.path.join(out_dir, "recovery_summary_by_landcover.csv"))

# print(f"✓ CSVs geladen:")
# print(f"  Major LC:     {len(df_traj_major):,} Zeilen")
# print(f"  LC×FireCount: {len(df_traj_fc):,} Zeilen")
# print(f"  LC×Ecoregion: {len(df_traj_eco):,} Zeilen")
# print(f"  Recovery:     {len(df_recovery):,} Zeilen")

# # --- Rekonstruiere Dictionaries aus CSVs ---
# rel_years = sorted(df_traj_major['Rel_Year'].unique())

# lc_major_results = {}
# for lc_class, grp in df_traj_major.groupby('LC_Major'):
#     grp = grp.sort_values('Rel_Year')
#     lc_major_results[lc_class] = {
#         'trajectory': grp['Woody_Cover_Mean'].tolist(),
#         'std': grp['Woody_Cover_Std'].tolist(),
#         'n_pixels': int(grp['N_Pixels'].iloc[0])
#     }

# lc_firecount_results = {}
# for (lc_class, fc), grp in df_traj_fc.groupby(['LC_Major', 'Fire_Count']):
#     grp = grp.sort_values('Rel_Year')
#     if lc_class not in lc_firecount_results:
#         lc_firecount_results[lc_class] = {}
#     lc_firecount_results[lc_class][int(fc)] = {
#         'trajectory': grp['Woody_Cover_Mean'].tolist(),
#         'std': grp['Woody_Cover_Std'].tolist(),
#         'n_pixels': int(grp['N_Pixels'].iloc[0])
#     }

# lc_eco_results = {}
# for (eco_code, lc_class), grp in df_traj_eco.groupby(['Ecoregion', 'LC_Major']):
#     grp = grp.sort_values('Rel_Year')
#     if eco_code not in lc_eco_results:
#         lc_eco_results[eco_code] = {}
#     lc_eco_results[eco_code][lc_class] = {
#         'trajectory': grp['Woody_Cover_Mean'].tolist(),
#         'std': grp['Woody_Cover_Std'].tolist(),
#         'n_pixels': int(grp['N_Pixels'].iloc[0])
#     }

# print(f"\n✓ Dictionaries rekonstruiert – Schritt 3 Plots können direkt ausgeführt werden!")
# print(f"  {len(lc_major_results)} Major LC Klassen")
# print(f"  {sum(len(v) for v in lc_firecount_results.values())} LC×FireCount Kombinationen")
# print(f"  {sum(len(v) for v in lc_eco_results.values())} LC×Ecoregion Kombinationen")

In [34]:
# === SCHRITT 3: VISUALISIERUNGEN ===

print("\n3. ERSTELLE VISUALISIERUNGEN")
print("-" * 70)

out_dir = os.path.join(workDir, "_Runs", "05_Landcover_integrated", "MODIS_IGBP_Analysis")
os.makedirs(out_dir, exist_ok=True)

# Farben für Landcover-Typen
LC_COLORS = {
    'Forests': '#228B22',
    'Shrublands': '#8B4513',
    'Savannas': '#DAA520',
    'Grasslands': '#90EE90',
    'Wetlands': '#4682B4',
    'Croplands': '#FFD700',
    'Barren/Ice': '#D3D3D3'
}

FIRE_COUNT_COLORS = {
    1: '#1f77b4',
    2: '#ff7f0e',
    3: '#2ca02c',
    4: '#d62728'
}

# =====================================================
# PLOT 1: Alle Landcover-Trajektorien (1 Fire) combined
# =====================================================
print("\nPlot 1: Alle LC-Trajektorien (1 Fire)...")

fig, ax = plt.subplots(figsize=(14, 8))

for lc_class, data in sorted(lc_major_results.items()):
    color = LC_COLORS.get(lc_class, '#808080')
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    
    ax.plot(rel_years, traj, marker='o', linewidth=2.5, 
            label=f"{lc_class} (n={data['n_pixels']:,})",
            color=color, alpha=0.9)
    
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    ax.fill_between(rel_years, lower, upper, alpha=0.15, color=color)

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Fire Year')
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)

ax.set_xlabel('Years Relative to Fire Event', fontsize=12)
ax.set_ylabel('Woody Cover (%)', fontsize=12)
ax.set_title('Post-Fire Woody Cover Trajectories by Pre-Fire Land Cover Type\n(Single Fire Events Only)', 
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5.5, 10.5)

ax.annotate('Pre-Fire', xy=(-3, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkgreen', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax.annotate('Fire\nYear', xy=(0.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkred', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.5))
ax.annotate('Recovery Period', xy=(5.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkblue', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plot1_path = os.path.join(out_dir, "lc_trajectories_1fire_all.png")
plt.savefig(plot1_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {plot1_path}")

# =====================================================
# PLOT 2: Einzelne Subplots pro Landcover-Typ (1 Fire)
# =====================================================
print("\nPlot 2: Subplots pro LC-Typ (1 Fire)...")

n_lc = len(lc_major_results)
n_cols = 3
n_rows = int(np.ceil(n_lc / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten()

for i, (lc_class, data) in enumerate(sorted(lc_major_results.items())):
    ax = axes[i]
    color = LC_COLORS.get(lc_class, '#808080')
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    
    y_min = max(0, np.nanmin(lower) - 5)
    y_max = min(100, np.nanmax(upper) + 5)
    
    ax.plot(rel_years, traj, marker='o', linewidth=2.5, color=color)
    ax.fill_between(rel_years, lower, upper, alpha=0.3, color=color)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    ax.set_ylim(y_min, y_max)
    ax.set_title(f"{lc_class}\n(n={data['n_pixels']:,} Events)", fontsize=11, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=9)
    ax.set_ylabel('Woody Cover (%)', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

for i in range(n_lc, len(axes)):
    axes[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Pre-Fire Land Cover (Single Fire Events)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plot2_path = os.path.join(out_dir, "lc_trajectories_1fire_panel.png")
plt.savefig(plot2_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {plot2_path}")

# =====================================================
# PLOT 3: Trajektorien nach Landcover × Fire Count
# =====================================================
print("\nPlot 3: LC × Fire Count Trajektorien...")

n_lc_fc = len([lc for lc in lc_firecount_results if lc_firecount_results[lc]])
n_cols = 2
n_rows = int(np.ceil(n_lc_fc / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 6 * n_rows))
axes = axes.flatten()

plot_idx = 0
for lc_class in sorted(lc_firecount_results.keys()):
    fc_data = lc_firecount_results[lc_class]
    if not fc_data:
        continue
    
    ax = axes[plot_idx]
    
    for fc, data in sorted(fc_data.items()):
        traj = np.array(data['trajectory'])
        std = np.array(data['std'])
        
        ax.plot(rel_years, traj, marker='o', linewidth=2.5,
                color=FIRE_COUNT_COLORS[fc],
                label=f'{fc} Fire(s) (n={data["n_pixels"]:,})',
                alpha=0.8)
        
        lower = np.maximum(traj - std, 0)
        upper = np.minimum(traj + std, 100)
        ax.fill_between(rel_years, lower, upper, alpha=0.15, color=FIRE_COUNT_COLORS[fc])
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.set_title(f"{lc_class}", fontsize=12, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=10)
    ax.set_ylabel('Woody Cover (%)', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)
    
    lc_color = LC_COLORS.get(lc_class, '#808080')
    for spine in ax.spines.values():
        spine.set_edgecolor(lc_color)
        spine.set_linewidth(2)
    
    plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Land Cover × Fire Count', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plot3_path = os.path.join(out_dir, "lc_trajectories_by_fire_count.png")
plt.savefig(plot3_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {plot3_path}")

# =====================================================
# PLOT 4: Heatmap Landcover × Ecoregion
# =====================================================
print("\nPlot 4: Heatmap LC × Ecoregion...")

# Berechne Recovery-Metriken für Heatmap
heatmap_data_loss = {}
heatmap_data_recovery = {}
heatmap_data_npixels = {}

for eco_code, lc_data in lc_eco_results.items():
    heatmap_data_loss[eco_code] = {}
    heatmap_data_recovery[eco_code] = {}
    heatmap_data_npixels[eco_code] = {}
    
    for lc_class, data in lc_data.items():
        traj = np.array(data['trajectory'])
        
        pre_fire = np.nanmean(traj[:5])   # -5 bis -1
        fire_year = traj[5]                # Jahr 0
        recovery_5yr = traj[10] if len(traj) > 10 else traj[-1]  # +5
        
        loss = pre_fire - fire_year if not np.isnan(fire_year) else np.nan
        rec_pct = ((recovery_5yr - fire_year) / loss * 100) if loss > 0 else np.nan
        
        heatmap_data_loss[eco_code][lc_class] = loss
        heatmap_data_recovery[eco_code][lc_class] = rec_pct
        heatmap_data_npixels[eco_code][lc_class] = data['n_pixels']

df_loss = pd.DataFrame(heatmap_data_loss).T
df_recovery = pd.DataFrame(heatmap_data_recovery).T
df_npixels = pd.DataFrame(heatmap_data_npixels).T

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

# Heatmap 1: Woody Loss
ax = axes[0]
sns.heatmap(df_loss, cmap='Reds', annot=True, fmt='.1f', ax=ax,
            cbar_kws={'label': 'Woody Cover Loss (%)'})
ax.set_title('Immediate Woody Cover Loss\n(Pre-Fire − Fire Year)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type')
ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Heatmap 2: Recovery %
ax = axes[1]
sns.heatmap(df_recovery, cmap='RdYlGn', annot=True, fmt='.0f', ax=ax,
            center=100, cbar_kws={'label': 'Recovery after 5 Years (%)'})
ax.set_title('Recovery after 5 Years (%)\n(100% = Full Recovery)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type')
ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Heatmap 3: Sample Size
ax = axes[2]
sns.heatmap(df_npixels, cmap='Blues', annot=True, fmt='.0f', ax=ax,
            cbar_kws={'label': 'Number of Pixels'})
ax.set_title('Sample Size\n(Pixels with Single Fire)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type')
ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Land Cover × Ecoregion: Fire Impact & Recovery Summary', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plot4_path = os.path.join(out_dir, "lc_ecoregion_heatmap.png")
plt.savefig(plot4_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {plot4_path}")

# =====================================================
# PLOT 5: Trajektorien pro Ecoregion × Landcover
# =====================================================
print("\nPlot 5: Trajektorien pro Ecoregion × Landcover...")

for eco_code, lc_data in sorted(lc_eco_results.items()):
    if not lc_data:
        continue
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for lc_class, data in sorted(lc_data.items()):
        color = LC_COLORS.get(lc_class, '#808080')
        traj = np.array(data['trajectory'])
        
        ax.plot(rel_years, traj, marker='o', linewidth=2.5,
                label=f"{lc_class} (n={data['n_pixels']:,})",
                color=color, alpha=0.9)
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    eco_name = eco_mapping.get(
        [k for k, v in eco_mapping.items() if v['code'] == eco_code][0], {}
    ).get('name', eco_code)
    
    ax.set_title(f'{eco_code} – {eco_name}\nWoody Cover Trajectories by Pre-Fire Land Cover', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire Event', fontsize=12)
    ax.set_ylabel('Woody Cover (%)', fontsize=12)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)
    
    plt.tight_layout()
    plot5_path = os.path.join(out_dir, f"lc_trajectories_{eco_code}.png")
    plt.savefig(plot5_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ {plot5_path}")


3. ERSTELLE VISUALISIERUNGEN
----------------------------------------------------------------------

Plot 1: Alle LC-Trajektorien (1 Fire)...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\lc_trajectories_1fire_all.png

Plot 2: Subplots pro LC-Typ (1 Fire)...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\lc_trajectories_1fire_panel.png

Plot 3: LC × Fire Count Trajektorien...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\lc_trajectories_by_fire_count.png

Plot 4: Heatmap LC × Ecoregion...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\lc_ecoregion_heatmap.png

Plot 5: Trajektorien pro Ecoregion × Landcover...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\lc_trajectories_Alpine.png
  ✓ \\141.20.140.57\DAS_gs

Landcover-Visualisierungen (Plots 1–5)
Plot 1 – lc_trajectories_1fire_all.png

Figure 1. Mean woody cover trajectories (±1 SD) relative to the year of a single fire event, stratified by pre-fire MODIS IGBP major land cover type across the entire study area. Year 0 denotes the fire year. Pre-fire conditions span years −5 to −1; the recovery period extends to +10 years post-fire. Sample sizes (n) indicate the number of single-fire pixels per land cover class.

Plot 2 – lc_trajectories_1fire_panel.png

Figure 2. Individual woody cover trajectories (mean ±1 SD) for each pre-fire major land cover type following a single fire event. Each panel shows the temporal dynamics from 5 years before to 10 years after the fire. Red dashed lines indicate the fire year. Coloured borders correspond to the respective land cover class.

Plot 3 – lc_trajectories_by_fire_count.png

Figure 3. Woody cover trajectories (mean ±1 SD) stratified by pre-fire land cover type and cumulative fire count (1–4 fires). Each panel represents one major land cover class; line colours indicate the number of fire events experienced by each pixel. Only the first fire event is used as the temporal reference point.

Plot 4 – lc_ecoregion_heatmap.png

Figure 4. Heatmaps summarising fire impact and recovery across ecoregions and pre-fire land cover types for single-fire pixels. (a) Immediate woody cover loss (pre-fire mean minus fire-year value, in percentage points). (b) Recovery after 5 years expressed as a percentage of the initial loss (100% = full recovery). (c) Sample size (number of single-fire pixels) per ecoregion × land cover combination.

Plot 5 – lc_trajectories_<ECO>.png (one per ecoregion)

Figure 5 (series). Woody cover trajectories by pre-fire land cover type within each ecoregion, shown for single fire events. Each figure corresponds to one ecoregion; line colours represent major land cover classes. Sample sizes are given in parentheses.

In [37]:
# === SCHRITT 3b: STATISTISCHE TESTS – LANDCOVER-VERGLEICHE ===
# Kruskal-Wallis H-Test + Dunn's Post-hoc auf Pixel-Ebene
# Testet: Unterscheiden sich Woody-Cover-Metriken signifikant zwischen LC-Typen?

print("\n3b. STATISTISCHE TESTS – LANDCOVER-VERGLEICHE")
print("-" * 70)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from scipy import stats

try:
    import scikit_posthocs as sp
    HAS_POSTHOC = True
    print("✓ scikit-posthocs verfügbar")
except ImportError:
    HAS_POSTHOC = False
    print("⚠️  scikit-posthocs nicht installiert – nur Kruskal-Wallis (kein Dunn Post-hoc)")
    print("    Installation: pip install scikit-posthocs")

# =====================================================
# HILFSFUNKTIONEN (gleich wie in 04_ecoregions)
# =====================================================

def extract_pixel_metrics_lc(woody, burned, fire_counts, first_fire_idx, mask,
                              years_woody, years_burned, nodata, max_samples=50000):
    """
    Extrahiert Pixel-Level Metriken für eine gegebene Maske.
    Nur Pixel mit genau 1 Brand.
    """
    single_fire_mask = (fire_counts == 1) & mask
    y_all, x_all = np.where(single_fire_mask)
    n_pixels = len(y_all)
    
    if n_pixels == 0:
        return None
    
    if n_pixels > max_samples:
        rng = np.random.default_rng(42)
        idx = rng.choice(n_pixels, max_samples, replace=False)
        y_all = y_all[idx]
        x_all = x_all[idx]
        n_pixels = max_samples
    
    offset = years_burned[0] - years_woody[0]
    fire_idx_subset = first_fire_idx[y_all, x_all]
    
    rel_years_extract = list(range(-5, 11))
    pixel_values = np.full((n_pixels, len(rel_years_extract)), np.nan)
    
    for j, rel_year in enumerate(rel_years_extract):
        woody_band_idx = fire_idx_subset + rel_year + offset - 1
        valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))
        if np.sum(valid) > 0:
            vals = woody[woody_band_idx[valid], y_all[valid], x_all[valid]].astype(float)
            vals[vals == nodata] = np.nan
            pixel_values[valid, j] = vals
    
    pre_fire_vals = pixel_values[:, 0:5]
    fire_year_val = pixel_values[:, 5]
    pre_fire_m1 = pixel_values[:, 4]
    recovery_5yr_val = pixel_values[:, 10]
    
    pre_fire_mean = np.nanmean(pre_fire_vals, axis=1)
    
    pre_fire_trend = np.full(n_pixels, np.nan)
    x_trend = np.arange(5)
    for i in range(n_pixels):
        vals_i = pre_fire_vals[i, :]
        valid_i = ~np.isnan(vals_i)
        if np.sum(valid_i) >= 3:
            slope, _, _, _, _ = stats.linregress(x_trend[valid_i], vals_i[valid_i])
            pre_fire_trend[i] = slope
    
    fire_loss = pre_fire_m1 - fire_year_val
    fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
    
    recovery_5yr_pct = np.where(
        fire_loss > 0,
        (recovery_5yr_val - fire_year_val) / fire_loss * 100,
        np.nan
    )
    
    return {
        'pre_fire_mean': pre_fire_mean,
        'pre_fire_trend': pre_fire_trend,
        'fire_loss': fire_loss,
        'fire_loss_pct': fire_loss_pct,
        'recovery_5yr_pct': recovery_5yr_pct,
        'woody_at_fire': fire_year_val,
        'n_pixels_total': int(np.sum(single_fire_mask)),
        'n_pixels_sampled': n_pixels
    }


def kruskal_wallis_with_effect_size(groups, group_names, metric_name):
    """Kruskal-Wallis H-Test mit Eta-squared Effektstärke."""
    clean_groups = []
    clean_names = []
    for g, name in zip(groups, group_names):
        valid = g[~np.isnan(g)]
        if len(valid) >= 5:
            clean_groups.append(valid)
            clean_names.append(name)
    
    if len(clean_groups) < 2:
        return None
    
    H_stat, p_value = stats.kruskal(*clean_groups)
    
    k = len(clean_groups)
    n_total = sum(len(g) for g in clean_groups)
    eta_squared = max(0, (H_stat - k + 1) / (n_total - k))
    
    if eta_squared < 0.01:
        effect_interp = "negligible"
    elif eta_squared < 0.06:
        effect_interp = "small"
    elif eta_squared < 0.14:
        effect_interp = "medium"
    else:
        effect_interp = "large"
    
    medians = {name: np.nanmedian(g) for name, g in zip(clean_names, clean_groups)}
    ns = {name: len(g) for name, g in zip(clean_names, clean_groups)}
    
    return {
        'metric': metric_name,
        'H_statistic': H_stat,
        'p_value': p_value,
        'eta_squared': eta_squared,
        'effect_size': effect_interp,
        'k_groups': k,
        'n_total': n_total,
        'medians': medians,
        'n_per_group': ns,
        'significant': p_value < 0.05,
        'groups': clean_groups,
        'group_names': clean_names
    }


METRICS_TO_TEST = ['pre_fire_mean', 'pre_fire_trend', 'fire_loss', 
                   'fire_loss_pct', 'recovery_5yr_pct']

METRIC_LABELS = {
    'pre_fire_mean': 'Pre-Fire Woody Cover (Mean, %)',
    'pre_fire_trend': 'Pre-Fire Trend (Slope, %/yr)',
    'fire_loss': 'Absolute Fire Loss (%)',
    'fire_loss_pct': 'Relative Fire Loss (%)',
    'recovery_5yr_pct': '5-Year Recovery (% of Loss)'
}

MAX_SAMPLES = 50000

# =====================================================
# TEST A: Vergleich zwischen Major LC-Klassen (Europa-weit)
# =====================================================

print("\n" + "=" * 70)
print("TEST A: Kruskal-Wallis – Major Landcover Klassen (Europa-weit)")
print("=" * 70)

print("\nExtrahiere Pixel-Level Metriken pro LC-Klasse...")

lc_pixel_metrics = {}

for major_id in tqdm(major_classes_present, desc="LC Klassen"):
    lc_name = MAJOR_LC_NAMES[major_id]
    mask = (pre_fire_lc_major_id == major_id)
    
    metrics = extract_pixel_metrics_lc(
        woody, burned, fire_counts, first_fire_idx, mask,
        years_woody, years_burned, nodata,
        max_samples=MAX_SAMPLES
    )
    
    if metrics is not None:
        lc_pixel_metrics[lc_name] = metrics
        print(f"  ✓ {lc_name}: {metrics['n_pixels_total']:,} Pixel "
              f"(sampled: {metrics['n_pixels_sampled']:,})")

# Kruskal-Wallis Tests
lc_names_ordered = sorted(lc_pixel_metrics.keys())

kw_results_lc = {}

for metric in METRICS_TO_TEST:
    groups = [lc_pixel_metrics[lc][metric] for lc in lc_names_ordered]
    result = kruskal_wallis_with_effect_size(groups, lc_names_ordered, metric)
    
    if result is not None:
        kw_results_lc[metric] = result
        sig_str = "***" if result['p_value'] < 0.001 else ("**" if result['p_value'] < 0.01 else ("*" if result['p_value'] < 0.05 else "n.s."))
        print(f"\n  {METRIC_LABELS[metric]}:")
        print(f"    H = {result['H_statistic']:.1f}, p = {result['p_value']:.2e} {sig_str}")
        print(f"    η² = {result['eta_squared']:.4f} ({result['effect_size']})")


# Dunn's Post-hoc
dunn_results_lc = {}

if HAS_POSTHOC:
    print("\n  Dunn's Post-hoc Tests (Bonferroni):")
    for metric in METRICS_TO_TEST:
        if metric not in kw_results_lc or not kw_results_lc[metric]['significant']:
            continue
        
        result = kw_results_lc[metric]
        all_values = []
        all_labels = []
        for g, name in zip(result['groups'], result['group_names']):
            all_values.extend(g.tolist())
            all_labels.extend([name] * len(g))
        
        # Erstelle DataFrame statt numpy-Arrays zu übergeben
        dunn_input_df = pd.DataFrame({
            'value': all_values,
            'group': all_labels
        })
        
        dunn_df = sp.posthoc_dunn(
            a=dunn_input_df,
            val_col='value',
            group_col='group',
            p_adjust='bonferroni'
        )
        dunn_results_lc[metric] = dunn_df
        
        n_pairs = dunn_df.shape[0] * (dunn_df.shape[0] - 1) // 2
        sig_pairs = np.sum(dunn_df.values[np.triu_indices_from(dunn_df.values, k=1)] < 0.05)
        print(f"    {metric}: {sig_pairs}/{n_pairs} signifikante Paare")

# =====================================================
# TEST B: LC-Vergleich INNERHALB jeder Ecoregion
# =====================================================

print("\n" + "=" * 70)
print("TEST B: Kruskal-Wallis – LC-Klassen innerhalb jeder Ecoregion")
print("=" * 70)

kw_results_lc_per_eco = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregions"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    
    eco_lc_metrics = {}
    
    for major_id in major_classes_present:
        lc_name = MAJOR_LC_NAMES[major_id]
        mask = (pre_fire_lc_major_id == major_id) & eco_mask
        
        metrics = extract_pixel_metrics_lc(
            woody, burned, fire_counts, first_fire_idx, mask,
            years_woody, years_burned, nodata,
            max_samples=20000  # Weniger Samples pro Eco×LC Kombination
        )
        
        if metrics is not None:
            eco_lc_metrics[lc_name] = metrics
    
    if len(eco_lc_metrics) < 2:
        continue
    
    kw_results_lc_per_eco[eco_code] = {}
    lc_names_eco = sorted(eco_lc_metrics.keys())
    
    for metric in METRICS_TO_TEST:
        groups = [eco_lc_metrics[lc][metric] for lc in lc_names_eco 
                  if lc in eco_lc_metrics]
        names = [lc for lc in lc_names_eco if lc in eco_lc_metrics]
        
        result = kruskal_wallis_with_effect_size(groups, names, metric)
        if result is not None:
            kw_results_lc_per_eco[eco_code][metric] = result

# Zusammenfassungstabelle: Ecoregion × Metrik → Effektstärke
print("\nZusammenfassung: Effektstärken (η²) der LC-Unterschiede pro Ecoregion:")
print(f"{'Ecoregion':<10}", end="")
for m in METRICS_TO_TEST:
    print(f"  {m[:15]:>15}", end="")
print()
print("-" * 90)

for eco_code in sorted(kw_results_lc_per_eco.keys()):
    print(f"{eco_code:<10}", end="")
    for metric in METRICS_TO_TEST:
        if metric in kw_results_lc_per_eco[eco_code]:
            r = kw_results_lc_per_eco[eco_code][metric]
            sig = "*" if r['significant'] else " "
            print(f"  {r['eta_squared']:>14.4f}{sig}", end="")
        else:
            print(f"  {'n/a':>15}", end="")
    print()

# =====================================================
# VISUALISIERUNGEN
# =====================================================

print("\nErstelle Visualisierungen...")

# --- Boxplots: Metriken nach LC-Klasse (Europa-weit) ---
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, metric in enumerate(METRICS_TO_TEST):
    ax = axes[i]
    
    box_data = []
    box_labels = []
    box_colors = []
    
    for lc_name in lc_names_ordered:
        if lc_name not in lc_pixel_metrics:
            continue
        vals = lc_pixel_metrics[lc_name][metric]
        valid = vals[~np.isnan(vals)]
        if len(valid) > 0:
            box_data.append(valid)
            box_labels.append(lc_name)
            box_colors.append(LC_COLORS.get(lc_name, '#808080'))
    
    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True,
                    showfliers=False, widths=0.6)
    
    for j, color in enumerate(box_colors):
        bp['boxes'][j].set_facecolor(color)
        bp['boxes'][j].set_alpha(0.6)
    
    ax.set_title(METRIC_LABELS[metric], fontsize=11, fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    if metric in kw_results_lc:
        kw = kw_results_lc[metric]
        sig_str = "***" if kw['p_value'] < 0.001 else ("**" if kw['p_value'] < 0.01 else ("*" if kw['p_value'] < 0.05 else "n.s."))
        ax.text(0.02, 0.98, f"KW: H={kw['H_statistic']:.0f}, η²={kw['eta_squared']:.3f} {sig_str}",
                transform=ax.transAxes, fontsize=8, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

axes[-1].axis('off')

plt.suptitle('Pixel-Level Metric Distributions by Pre-Fire Land Cover (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
boxplot_lc_path = os.path.join(out_dir, "statistical_tests_landcover_boxplots.png")
plt.savefig(boxplot_lc_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {boxplot_lc_path}")

# --- Dunn Heatmaps ---
if HAS_POSTHOC and dunn_results_lc:
    n_metrics_sig = len(dunn_results_lc)
    fig, axes = plt.subplots(1, n_metrics_sig, figsize=(7 * n_metrics_sig, 6))
    if n_metrics_sig == 1:
        axes = [axes]
    
    plot_i = 0
    for metric in METRICS_TO_TEST:
        if metric not in dunn_results_lc:
            continue
        ax = axes[plot_i]
        dunn_df = dunn_results_lc[metric]
        
        log_p = -np.log10(dunn_df.values.astype(float))
        log_p = np.clip(log_p, 0, 50)
        
        annot = dunn_df.copy()
        for r in annot.index:
            for c in annot.columns:
                p = dunn_df.loc[r, c]
                if r == c:
                    annot.loc[r, c] = ""
                elif p < 0.001:
                    annot.loc[r, c] = "***"
                elif p < 0.01:
                    annot.loc[r, c] = "**"
                elif p < 0.05:
                    annot.loc[r, c] = "*"
                else:
                    annot.loc[r, c] = "n.s."
        
        sns.heatmap(
            pd.DataFrame(log_p, index=dunn_df.index, columns=dunn_df.columns),
            cmap='RdYlGn_r', annot=annot.values, fmt='',
            ax=ax, square=True, linewidths=0.5,
            cbar_kws={'label': '-log₁₀(p)'},
            vmin=0, vmax=10
        )
        ax.set_title(f'{METRIC_LABELS[metric]}\n(Dunn Post-hoc, Bonferroni)',
                     fontsize=10, fontweight='bold')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
        plot_i += 1
    
    plt.suptitle("Pairwise Land Cover Comparisons – Dunn's Post-hoc Test",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    heatmap_lc_path = os.path.join(out_dir, "statistical_tests_landcover_dunn_heatmaps.png")
    plt.savefig(heatmap_lc_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ {heatmap_lc_path}")

# --- Heatmap: Effektstärke Ecoregion × Metrik ---
if kw_results_lc_per_eco:
    eco_codes_sorted = sorted(kw_results_lc_per_eco.keys())
    
    eta_matrix = pd.DataFrame(index=eco_codes_sorted, columns=METRICS_TO_TEST, dtype=float)
    sig_matrix = pd.DataFrame(index=eco_codes_sorted, columns=METRICS_TO_TEST, dtype=str)
    
    for eco_code in eco_codes_sorted:
        for metric in METRICS_TO_TEST:
            if metric in kw_results_lc_per_eco[eco_code]:
                r = kw_results_lc_per_eco[eco_code][metric]
                eta_matrix.loc[eco_code, metric] = r['eta_squared']
                sig_str = "***" if r['p_value'] < 0.001 else ("**" if r['p_value'] < 0.01 else ("*" if r['p_value'] < 0.05 else ""))
                sig_matrix.loc[eco_code, metric] = f"{r['eta_squared']:.3f}{sig_str}"
            else:
                sig_matrix.loc[eco_code, metric] = ""
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    eta_numeric = eta_matrix.astype(float)
    
    sns.heatmap(
        eta_numeric, cmap='YlOrRd', annot=sig_matrix.values, fmt='',
        ax=ax, linewidths=0.5,
        cbar_kws={'label': 'Effect Size (η²)'},
        vmin=0, vmax=0.3
    )
    
    ax.set_title('Effect Size of Land Cover Differences Within Each Ecoregion\n'
                 '(Kruskal-Wallis η², * p<.05, ** p<.01, *** p<.001)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Metric')
    ax.set_ylabel('Ecoregion')
    
    short_labels = [m.replace('_', '\n') for m in METRICS_TO_TEST]
    ax.set_xticklabels(short_labels, rotation=45, ha='right')
    
    plt.tight_layout()
    eta_heatmap_path = os.path.join(out_dir, "statistical_tests_lc_x_ecoregion_effect_sizes.png")
    plt.savefig(eta_heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ {eta_heatmap_path}")

# =====================================================
# CSV EXPORT
# =====================================================

print("\nSpeichere statistische Testergebnisse...")

# CSV: KW Zusammenfassung (LC Europa-weit)
kw_rows = []
for metric in METRICS_TO_TEST:
    if metric in kw_results_lc:
        kw = kw_results_lc[metric]
        kw_rows.append({
            'Metric': metric,
            'Metric_Label': METRIC_LABELS[metric],
            'H_Statistic': kw['H_statistic'],
            'p_value': kw['p_value'],
            'Eta_Squared': kw['eta_squared'],
            'Effect_Size': kw['effect_size'],
            'k_Groups': kw['k_groups'],
            'n_Total': kw['n_total'],
            'Significant_0.05': kw['significant']
        })

df_kw_lc = pd.DataFrame(kw_rows)
kw_lc_csv = os.path.join(out_dir, "statistical_tests_landcover_kruskal_wallis.csv")
df_kw_lc.to_csv(kw_lc_csv, index=False)
print(f"  ✓ {kw_lc_csv}")

# CSV: KW innerhalb Ecoregions
eco_kw_rows = []
for eco_code in sorted(kw_results_lc_per_eco.keys()):
    for metric in METRICS_TO_TEST:
        if metric in kw_results_lc_per_eco[eco_code]:
            r = kw_results_lc_per_eco[eco_code][metric]
            eco_kw_rows.append({
                'Ecoregion': eco_code,
                'Metric': metric,
                'H_Statistic': r['H_statistic'],
                'p_value': r['p_value'],
                'Eta_Squared': r['eta_squared'],
                'Effect_Size': r['effect_size'],
                'k_Groups': r['k_groups'],
                'n_Total': r['n_total'],
                'Significant_0.05': r['significant']
            })

df_eco_kw = pd.DataFrame(eco_kw_rows)
eco_kw_csv = os.path.join(out_dir, "statistical_tests_lc_x_ecoregion_kruskal_wallis.csv")
df_eco_kw.to_csv(eco_kw_csv, index=False)
print(f"  ✓ {eco_kw_csv}")

# CSV: Dunn Post-hoc p-Werte (LC Europa-weit)
if HAS_POSTHOC and dunn_results_lc:
    for metric, dunn_df in dunn_results_lc.items():
        dunn_csv = os.path.join(out_dir, f"statistical_tests_landcover_dunn_{metric}.csv")
        dunn_df.to_csv(dunn_csv)
        print(f"  ✓ {dunn_csv}")

# CSV: Mediane pro LC-Klasse
median_rows = []
for lc_name in lc_names_ordered:
    if lc_name not in lc_pixel_metrics:
        continue
    row = {'LC_Major': lc_name, 'N_Pixels_Sampled': lc_pixel_metrics[lc_name]['n_pixels_sampled']}
    for metric in METRICS_TO_TEST:
        vals = lc_pixel_metrics[lc_name][metric]
        row[f'{metric}_median'] = np.nanmedian(vals)
        row[f'{metric}_iqr'] = stats.iqr(vals[~np.isnan(vals)]) if np.sum(~np.isnan(vals)) > 0 else np.nan
    median_rows.append(row)

df_medians_lc = pd.DataFrame(median_rows)
medians_lc_csv = os.path.join(out_dir, "statistical_tests_landcover_medians.csv")
df_medians_lc.to_csv(medians_lc_csv, index=False)
print(f"  ✓ {medians_lc_csv}")

print(f"\n✓ Alle statistischen Tests für Landcover abgeschlossen!")


3b. STATISTISCHE TESTS – LANDCOVER-VERGLEICHE
----------------------------------------------------------------------
✓ scikit-posthocs verfügbar

TEST A: Kruskal-Wallis – Major Landcover Klassen (Europa-weit)

Extrahiere Pixel-Level Metriken pro LC-Klasse...


LC Klassen:   0%|          | 0/7 [00:00<?, ?it/s]C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:77: RuntimeWarning: divide by zero encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:77: RuntimeWarning: invalid value encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:81: RuntimeWarning: divide by zero encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100,
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:81: RuntimeWarning: invalid value encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100,
LC Klassen:  14%|█▍        | 1/7 [00:08<00:53,  8.97s/it]

  ✓ Forests: 157,808 Pixel (sampled: 50,000)


LC Klassen:  29%|██▊       | 2/7 [00:11<00:26,  5.35s/it]

  ✓ Shrublands: 12,943 Pixel (sampled: 12,943)


...


  Relative Fire Loss (%):
    H = 904.4, p = 4.30e-192 ***
    η² = 0.0042 (negligible)

  5-Year Recovery (% of Loss):
    H = 3533.1, p = 0.00e+00 ***
    η² = 0.0595 (small)

  Dunn's Post-hoc Tests (Bonferroni):
    pre_fire_mean: 21/21 signifikante Paare
    pre_fire_trend: 19/21 signifikante Paare
    fire_loss: 16/21 signifikante Paare
    fire_loss_pct: 15/21 signifikante Paare
    recovery_5yr_pct: 18/21 signifikante Paare

TEST B: Kruskal-Wallis – LC-Klassen innerhalb jeder Ecoregion


Ecoregions:   0%|          | 0/11 [00:00<?, ?it/s]C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:81: RuntimeWarning: invalid value encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100,
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:77: RuntimeWarning: divide by zero encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:77: RuntimeWarning: invalid value encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:81: RuntimeWarning: divide by zero encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100,
Ecoregions: 100%|██████████| 11/11 [02:26<00:00, 13.35s/it]



Zusammenfassung: Effektstärken (η²) der LC-Unterschiede pro Ecoregion:
Ecoregion     pre_fire_mean   pre_fire_trend        fire_loss    fire_loss_pct  recovery_5yr_pc
------------------------------------------------------------------------------------------
Alpine              0.5219*          0.0195*          0.0006*          0.0009*          0.0079*
Anatolian           0.0254*          0.0036*          0.0015*          0.0020*          0.0034*
Arctic              0.6460*          0.0387*          0.0054           0.0005           0.1377*
Atlantic            0.5280*          0.0038*          0.0048*          0.0044*          0.0241*
BlackSea            0.3310*          0.0366*          0.0076*          0.0146*          0.0137*
Boreal              0.4279*          0.0047*          0.0018*          0.0025*          0.0406*
Continental          0.5911*          0.0194*          0.0074*          0.0059*          0.0998*
Mediterranean          0.5787*          0.0168*          0.0088*    

C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:367: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:367: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:367: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))


  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\statistical_tests_landcover_dunn_heatmaps.png
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\statistical_tests_lc_x_ecoregion_effect_sizes.png

Speichere statistische Testergebnisse...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\statistical_tests_landcover_kruskal_wallis.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\statistical_tests_lc_x_ecoregion_kruskal_wallis.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\statistical_tests_landcover_dunn_pre_fire_mean.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\statistical_tests_landcover_dunn_pre_fire_trend.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butze

Schritt 3b: Statistische Tests – Landcover (Boxplots & Heatmaps)
statistical_tests_landcover_boxplots.png

Figure 6. Box plot distributions of pixel-level fire impact metrics by pre-fire major land cover class (single fire events, Europe-wide). Metrics shown are: pre-fire mean woody cover, pre-fire trend (slope), absolute and relative fire loss, and 5-year recovery (% of loss). Boxes are coloured by land cover type; outliers are suppressed. Kruskal–Wallis H-statistic, effect size (η²), and significance level are annotated per panel.

statistical_tests_landcover_dunn_heatmaps.png

Figure 7. Pairwise post-hoc comparisons (Dunn's test with Bonferroni correction) between pre-fire major land cover classes for each metric where the Kruskal–Wallis test was significant (p < 0.05). Cell colours represent −log₁₀(p); annotations indicate significance levels (*** p < 0.001, ** p < 0.01, * p < 0.05, n.s. = not significant).

statistical_tests_lc_x_ecoregion_effect_sizes.png

Figure 8. Effect sizes (η²) of Kruskal–Wallis tests comparing pre-fire land cover classes within each ecoregion. Rows represent ecoregions; columns represent fire impact metrics. Significance levels are annotated (* p < 0.05, ** p < 0.01, *** p < 0.001). Warmer colours indicate larger effect sizes, reflecting stronger land cover differentiation within a given ecoregion.

In [38]:
# === SCHRITT 3c: ECOREGION-VISUALISIERUNGEN ===
# (Ersetzt Schritte 7–15 aus 04_ecoregions_MBA_analysis.ipynb)

print("\n3c. ECOREGION-VISUALISIERUNGEN")
print("-" * 70)

# =====================================================
# PLOT E1: Alle Ecoregion-Trajektorien (1 Fire)
# =====================================================
print("\nPlot E1: Alle Ecoregion-Trajektorien (1 Fire)...")

fig, ax = plt.subplots(figsize=(14, 8))

for eco_code, data in sorted(eco_results.items()):
    color = data['hex_color']
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    
    ax.plot(rel_years, traj, marker='o', linewidth=2.5,
            label=f"{eco_code} – {data['name']} (n={data['n_pixels']:,})",
            color=color, alpha=0.9)
    
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    ax.fill_between(rel_years, lower, upper, alpha=0.1, color=color)

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Fire Year')
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)

ax.annotate('Pre-Fire', xy=(-3, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkgreen', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax.annotate('Fire\nYear', xy=(0.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkred', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.5))
ax.annotate('Recovery', xy=(5.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkblue', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

ax.set_xlabel('Years Relative to Fire Event', fontsize=12)
ax.set_ylabel('Woody Cover (%)', fontsize=12)
ax.set_title('Post-Fire Woody Cover Trajectories by Ecoregion\n(Single Fire Events Only)',
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5.5, 10.5)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "eco_trajectories_1fire_all.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_trajectories_1fire_all.png")

# =====================================================
# PLOT E2: Subplots pro Ecoregion (1 Fire)
# =====================================================
print("\nPlot E2: Subplots pro Ecoregion...")

n_eco = len(eco_results)
n_cols = 3
n_rows = int(np.ceil(n_eco / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten()

for i, (eco_code, data) in enumerate(sorted(eco_results.items())):
    ax = axes[i]
    color = data['hex_color']
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    
    ax.plot(rel_years, traj, marker='o', linewidth=2.5, color=color)
    ax.fill_between(rel_years, lower, upper, alpha=0.3, color=color)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    ax.set_ylim(max(0, np.nanmin(lower) - 5), min(100, np.nanmax(upper) + 5))
    ax.set_title(f"{eco_code} – {data['name']}\n(n={data['n_pixels']:,})",
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=9)
    ax.set_ylabel('Woody Cover (%)', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

for i in range(n_eco, len(axes)):
    axes[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Ecoregion (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "eco_trajectories_1fire_panel.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_trajectories_1fire_panel.png")

# =====================================================
# PLOT E3: Ecoregion × Fire Count
# =====================================================
print("\nPlot E3: Ecoregion × Fire Count...")

eco_with_fc = {e: d for e, d in eco_firecount_results.items() if d}
n_eco_fc = len(eco_with_fc)
n_cols = 3
n_rows = int(np.ceil(n_eco_fc / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

plot_idx = 0
for eco_code in sorted(eco_with_fc.keys()):
    fc_data = eco_with_fc[eco_code]
    ax = axes[plot_idx]
    eco_name = eco_results.get(eco_code, {}).get('name', eco_code)
    
    for fc, data in sorted(fc_data.items()):
        traj = np.array(data['trajectory'])
        std = np.array(data['std'])
        
        ax.plot(rel_years, traj, marker='o', linewidth=2.5,
                color=FIRE_COUNT_COLORS[fc],
                label=f'{fc} Fire(s) (n={data["n_pixels"]:,})', alpha=0.8)
        
        lower = np.maximum(traj - std, 0)
        upper = np.minimum(traj + std, 100)
        ax.fill_between(rel_years, lower, upper, alpha=0.15, color=FIRE_COUNT_COLORS[fc])
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.set_title(f"{eco_code} – {eco_name}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=9)
    ax.set_ylabel('Woody Cover (%)', fontsize=9)
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)
    
    eco_color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
    for spine in ax.spines.values():
        spine.set_edgecolor(eco_color)
        spine.set_linewidth(2)
    
    plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Ecoregion × Fire Count',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "eco_trajectories_by_fire_count.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_trajectories_by_fire_count.png")

# =====================================================
# PLOT E4: Jährlich verbrannte Fläche (gesamt)
# =====================================================
print("\nPlot E4: Jährlich verbrannte Fläche...")

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(burned_area_total['Year'], burned_area_total['Burned_km2'],
       color='#d62728', alpha=0.8, edgecolor='darkred', linewidth=0.5)

mean_val = burned_area_total['Burned_km2'].mean()
ax.axhline(y=mean_val, color='black', linestyle='--', linewidth=1.5,
           label=f'Mean: {mean_val:,.0f} km²/yr')

# Werte auf Bars
for _, row in burned_area_total.iterrows():
    ax.text(row['Year'], row['Burned_km2'], f'{row["Burned_km2"]:.0f}',
            ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Burned Area (km²)', fontsize=12)
ax.set_title('Annual Burned Area – Entire Study Area', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "burned_area_annual_total.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ burned_area_annual_total.png")

# =====================================================
# PLOT E5: Verbrannte Fläche pro Ecoregion (absolut + prozentual)
# =====================================================
print("\nPlot E5: Burned Area pro Ecoregion (absolut + %)...")

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Pivot für Line-Plots
pivot_abs = burned_area_eco.pivot_table(
    index='Year', columns='Ecoregion', values='Burned_km2', fill_value=0
)
pivot_pct = burned_area_eco.pivot_table(
    index='Year', columns='Ecoregion', values='Burned_Fraction', fill_value=0
) * 100  # → Prozent

for eco_code in sorted(pivot_abs.columns):
    color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
    axes[0].plot(pivot_abs.index, pivot_abs[eco_code], marker='o', linewidth=2,
                 label=eco_code, color=color, alpha=0.85)
    axes[1].plot(pivot_pct.index, pivot_pct[eco_code], marker='o', linewidth=2,
                 label=eco_code, color=color, alpha=0.85)

axes[0].set_title('Annual Burned Area per Ecoregion (absolute)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Burned Area (km²)', fontsize=12)
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Annual Burned Area per Ecoregion (% of Region)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Burned Area (%)', fontsize=12)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('Year', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "burned_area_by_ecoregion.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ burned_area_by_ecoregion.png")

# =====================================================
# PLOT E6: Fire Season Length – Range + Distribution + Trend
# =====================================================
print("\nPlot E6: Fire Season Length (3 Subplots)...")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# E6a) Range Plot (Min-Mean-Max)
ax = axes[0]
eco_codes_sorted = sorted(fire_stats_summary['Ecoregion'].values)
y_pos = np.arange(len(eco_codes_sorted))

for i, eco_code in enumerate(eco_codes_sorted):
    row = fire_stats_summary[fire_stats_summary['Ecoregion'] == eco_code].iloc[0]
    color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
    ax.plot([row['Season_Length_Min'], row['Season_Length_Max']], [i, i],
            'o-', linewidth=2, markersize=8, color=color)
    ax.plot(row['Season_Length_Mean'], i, 'D', markersize=10, color='darkred', zorder=3)

ax.set_yticks(y_pos)
ax.set_yticklabels(eco_codes_sorted)
ax.set_xlabel('Fire Season Length (Months)', fontsize=11)
ax.set_title('Fire Season Length Range\n(Min-Mean-Max)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# E6b) Box Plot Distribution
ax = axes[1]
season_box_data = []
season_box_labels = []
season_box_colors = []

for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    
    all_season = mba_season[:, eco_mask].flatten()
    all_season = all_season[(all_season > 0) & (~np.isnan(all_season))]
    
    if len(all_season) > 0:
        season_box_data.append(all_season)
        season_box_labels.append(eco_code)
        season_box_colors.append(eco_results.get(eco_code, {}).get('hex_color', '#808080'))

bp = ax.boxplot(season_box_data, labels=season_box_labels, patch_artist=True,
                showfliers=False, vert=False)
for j, color in enumerate(season_box_colors):
    bp['boxes'][j].set_facecolor(color)
    bp['boxes'][j].set_alpha(0.6)

ax.set_xlabel('Fire Season Length (Months)', fontsize=11)
ax.set_title('Fire Season Length Distribution', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# E6c) Temporal Trend
ax = axes[2]

yearly_season_means = []
yearly_season_max = []
for yr_idx in range(len(years_burned)):
    vals = mba_season[yr_idx]
    valid = vals[(vals > 0) & (~np.isnan(vals))]
    yearly_season_means.append(float(np.mean(valid)) if len(valid) > 0 else np.nan)
    yearly_season_max.append(float(np.max(valid)) if len(valid) > 0 else np.nan)

ax.plot(years_burned, yearly_season_means, marker='o', linewidth=2, color='orangered', label='Mean')
ax.plot(years_burned, yearly_season_max, marker='s', linewidth=2, color='darkred',
        label='Maximum', linestyle='--')
ax.fill_between(years_burned, yearly_season_means, alpha=0.3, color='orangered')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Season Length (Months)', fontsize=11)
ax.set_title('Temporal Trend: Season Length', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Fire Season Length Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "fire_season_length_analysis.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ fire_season_length_analysis.png")

# =====================================================
# PLOT E7: Fire Count – Range + Distribution + Trend
# =====================================================
print("\nPlot E7: Fire Count (3 Subplots)...")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# E7a) Range
ax = axes[0]
for i, eco_code in enumerate(eco_codes_sorted):
    row = fire_stats_summary[fire_stats_summary['Ecoregion'] == eco_code].iloc[0]
    color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
    ax.plot([row['Fire_Count_Min'], row['Fire_Count_Max']], [i, i],
            'o-', linewidth=2, markersize=8, color=color)
    ax.plot(row['Fire_Count_Mean'], i, 'D', markersize=10, color='darkorange', zorder=3)

ax.set_yticks(y_pos)
ax.set_yticklabels(eco_codes_sorted)
ax.set_xlabel('Number of Fire Events', fontsize=11)
ax.set_title('Fire Count Range\n(Min-Mean-Max)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# E7b) Box Plot
ax = axes[1]
count_box_data = []
count_box_labels = []
count_box_colors = []

for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    
    all_count = mba_count[:, eco_mask].flatten()
    all_count = all_count[(all_count > 0) & (~np.isnan(all_count))]
    
    if len(all_count) > 0:
        count_box_data.append(all_count)
        count_box_labels.append(eco_code)
        count_box_colors.append(eco_results.get(eco_code, {}).get('hex_color', '#808080'))

bp = ax.boxplot(count_box_data, labels=count_box_labels, patch_artist=True,
                showfliers=False, vert=False)
for j, color in enumerate(count_box_colors):
    bp['boxes'][j].set_facecolor(color)
    bp['boxes'][j].set_alpha(0.6)

ax.set_xlabel('Number of Fire Events', fontsize=11)
ax.set_title('Fire Count Distribution', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# E7c) Temporal Trend
ax = axes[2]
yearly_count_means = []
yearly_count_max = []

for yr_idx in range(len(years_burned)):
    vals = mba_count[yr_idx]
    valid = vals[(vals > 0) & (~np.isnan(vals))]
    yearly_count_means.append(float(np.mean(valid)) if len(valid) > 0 else np.nan)
    yearly_count_max.append(float(np.max(valid)) if len(valid) > 0 else np.nan)

ax.plot(years_burned, yearly_count_means, marker='o', linewidth=2, color='darkorange', label='Mean')
ax.plot(years_burned, yearly_count_max, marker='s', linewidth=2, color='chocolate',
        label='Maximum', linestyle='--')
ax.fill_between(years_burned, yearly_count_means, alpha=0.3, color='orange')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Fire Count', fontsize=11)
ax.set_title('Temporal Trend: Fire Count', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Fire Count Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "fire_count_analysis.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ fire_count_analysis.png")

# =====================================================
# PLOT E8: Season Length × Fire Count
# =====================================================
print("\nPlot E8: Season Length abhängig von Fire Count...")

fig, ax = plt.subplots(figsize=(14, 8))

bp_data = [season_by_fc[c]['values'] for c in sorted(season_by_fc.keys())]
bp_labels = [f"{c} Fires\n(n={season_by_fc[c]['n']:,})" for c in sorted(season_by_fc.keys())]

bp = ax.boxplot(bp_data, labels=bp_labels, patch_artist=True)
colors_gradient = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(bp_data)))
for patch, color in zip(bp['boxes'], colors_gradient):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for i, fc_val in enumerate(sorted(season_by_fc.keys())):
    ax.text(i + 1, season_by_fc[fc_val]['mean'], f'{season_by_fc[fc_val]["mean"]:.1f}',
            ha='center', va='bottom', fontweight='bold', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

ax.set_xlabel('Number of Fire Events', fontsize=12)
ax.set_ylabel('Fire Season Length (Months)', fontsize=12)
ax.set_title('Fire Season Length by Number of Fire Events', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "fire_season_length_by_count.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ fire_season_length_by_count.png")

# =====================================================
# PLOT E9: Recovery Rates (5 & 10 Jahre) – Heatmap
# =====================================================
print("\nPlot E9: Recovery Rates Heatmap...")

# Berechne Recovery-Metriken pro Ecoregion
eco_recovery_rows = []

for eco_code, data in eco_results.items():
    traj = np.array(data['trajectory'])
    pre_fire = np.nanmean(traj[:5])
    fire_year_val = traj[5]
    loss = pre_fire - fire_year_val
    
    for years_after in [1, 3, 5, 10]:
        idx = 5 + years_after
        if idx < len(traj):
            rec_val = traj[idx]
            rec_pct = ((rec_val - fire_year_val) / loss * 100) if loss > 0 else np.nan
        else:
            rec_val = np.nan
            rec_pct = np.nan
        
        eco_recovery_rows.append({
            'Ecoregion': eco_code,
            'Years_After': years_after,
            'Pre_Fire_Cover': pre_fire,
            'Fire_Year_Cover': fire_year_val,
            'Woody_Loss': loss,
            'Recovery_Cover': rec_val,
            'Recovery_Percent': rec_pct,
            'N_Pixels': data['n_pixels']
        })

df_eco_recovery = pd.DataFrame(eco_recovery_rows)

# Heatmap: Ecoregion × Years_After → Recovery %
pivot_rec = df_eco_recovery.pivot_table(
    index='Ecoregion', columns='Years_After', values='Recovery_Percent'
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_rec, cmap='RdYlGn', annot=True, fmt='.0f', ax=ax,
            center=100, linewidths=0.5,
            cbar_kws={'label': 'Recovery (% of Loss)'})
ax.set_title('Woody Cover Recovery by Ecoregion\n(% of Immediate Loss Recovered)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Years After Fire')
ax.set_ylabel('Ecoregion')

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "eco_recovery_heatmap.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_recovery_heatmap.png")

# =====================================================
# PLOT E10: Summary Heatmap (Eco × Fire Count × Metriken)
# =====================================================
print("\nPlot E10: Comprehensive Summary Heatmap...")

# Berechne Summary-Metriken pro Eco × Fire Count
summary_rows = []

for eco_code, fc_data in eco_firecount_results.items():
    eco_name = eco_results.get(eco_code, {}).get('name', eco_code)
    eco_total_pixels = int(np.sum(eco_raster == eco_results.get(eco_code, {}).get('eco_id', 0)))
    
    for fc, data in fc_data.items():
        traj = np.array(data['trajectory'])
        pre_fire = np.nanmean(traj[:5])
        fire_year_val = traj[5]
        loss = pre_fire - fire_year_val
        
        # 5yr und 10yr Recovery
        rec_5yr = traj[10] if len(traj) > 10 else np.nan
        rec_5yr_pct = ((rec_5yr - fire_year_val) / loss * 100) if loss > 0 else np.nan
        
        rec_10yr = traj[15] if len(traj) > 15 else np.nan
        rec_10yr_pct = ((rec_10yr - fire_year_val) / loss * 100) if loss > 0 else np.nan
        
        summary_rows.append({
            'Ecoregion': eco_code,
            'Ecoregion_Name': eco_name,
            'Fire_Count': fc,
            'N_Pixels': data['n_pixels'],
            'Area_km2': data['n_pixels'] * pixel_area_km2,
            'Pre_Fire_Cover': pre_fire,
            'Fire_Year_Cover': fire_year_val,
            'Woody_Loss': loss,
            'Recovery_5yr_Pct': rec_5yr_pct,
            'Recovery_10yr_Pct': rec_10yr_pct
        })

df_summary = pd.DataFrame(summary_rows)

# 4-Panel Heatmap
metrics_hm = ['Area_km2', 'Woody_Loss', 'Recovery_5yr_Pct', 'Recovery_10yr_Pct']
titles_hm = ['Affected Area (km²)', 'Immediate Loss (%)', 
             '5-Year Recovery (%)', '10-Year Recovery (%)']
cmaps_hm = ['YlOrRd', 'Reds', 'RdYlGn', 'RdYlGn']
centers_hm = [None, None, 100, 100]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for idx, (metric, title, cmap, center) in enumerate(zip(metrics_hm, titles_hm, cmaps_hm, centers_hm)):
    ax = axes[idx]
    
    pivot = df_summary.pivot_table(
        index='Fire_Count', columns='Ecoregion', values=metric
    )
    
    sns.heatmap(pivot, cmap=cmap, annot=True, 
                fmt='.1f' if 'Area' in metric else '.0f',
                ax=ax, linewidths=0.5, center=center,
                cbar_kws={'label': title})
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Fire Count')
    ax.set_yticklabels([f'{int(fc)} Fire(s)' for fc in pivot.index])

plt.suptitle('Comprehensive Summary: Ecoregion × Fire Count',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "eco_summary_heatmap.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_summary_heatmap.png")

print(f"\n✓ Alle Ecoregion-Visualisierungen erstellt!")


3c. ECOREGION-VISUALISIERUNGEN
----------------------------------------------------------------------

Plot E1: Alle Ecoregion-Trajektorien (1 Fire)...
  ✓ eco_trajectories_1fire_all.png

Plot E2: Subplots pro Ecoregion...
  ✓ eco_trajectories_1fire_panel.png

Plot E3: Ecoregion × Fire Count...
  ✓ eco_trajectories_by_fire_count.png

Plot E4: Jährlich verbrannte Fläche...
  ✓ burned_area_annual_total.png

Plot E5: Burned Area pro Ecoregion (absolut + %)...
  ✓ burned_area_by_ecoregion.png

Plot E6: Fire Season Length (3 Subplots)...
  ✓ fire_season_length_analysis.png

Plot E7: Fire Count (3 Subplots)...
  ✓ fire_count_analysis.png

Plot E8: Season Length abhängig von Fire Count...
  ✓ fire_season_length_by_count.png

Plot E9: Recovery Rates Heatmap...
  ✓ eco_recovery_heatmap.png

Plot E10: Comprehensive Summary Heatmap...
  ✓ eco_summary_heatmap.png

✓ Alle Ecoregion-Visualisierungen erstellt!


Schritt 3c: Ecoregion-Visualisierungen (Plots E1–E10)
E1 – eco_trajectories_1fire_all.png

Figure 9. Mean woody cover trajectories (±1 SD) relative to a single fire event, stratified by ecoregion. Line colours correspond to the ecoregion colour scheme. Year 0 denotes the fire year; the pre-fire window spans years −5 to −1 and the recovery period extends to +10 years.

E2 – eco_trajectories_1fire_panel.png

Figure 10. Individual woody cover trajectories (mean ±1 SD) for each ecoregion following a single fire event. Coloured panel borders correspond to the ecoregion colour scheme. Y-axis ranges are adjusted per panel to highlight regional dynamics.

E3 – eco_trajectories_by_fire_count.png

Figure 11. Woody cover trajectories stratified by ecoregion and cumulative fire count (1–4 fires). Each panel represents one ecoregion; line colours indicate the number of fire events. The first fire event serves as the temporal reference point (year 0).

E4 – burned_area_annual_total.png

Figure 12. Annual burned area (km²) across the entire study area from 2000 to 2025. The dashed horizontal line indicates the long-term mean. Individual bar labels show the annual burned area value.

E5 – burned_area_by_ecoregion.png

Figure 13. Annual burned area by ecoregion shown as (a) absolute values in km² and (b) as a fraction of the total ecoregion area (%). Line colours correspond to the ecoregion colour scheme.

E6 – fire_season_length_analysis.png

Figure 14. Fire season length analysis across ecoregions. (a) Range plot showing minimum, mean (diamond), and maximum season length per ecoregion. (b) Box plot distribution of fire season length across all burned pixels per ecoregion. (c) Temporal trend of mean and maximum fire season length across the entire study area (2000–2025).

E7 – fire_count_analysis.png

Figure 15. Fire count analysis across ecoregions. (a) Range plot showing minimum, mean (diamond), and maximum number of fire events per ecoregion. (b) Box plot distribution of fire counts across all burned pixels per ecoregion. (c) Temporal trend of mean and maximum fire counts across the study area (2000–2025).

E8 – fire_season_length_by_count.png

Figure 16. Fire season length as a function of cumulative fire count. Box plots show the distribution of season length (months) for pixels that experienced 1, 2, 3, or more fire events. Mean values are annotated above each box. Colours follow a sequential gradient from yellow to red.

E9 – eco_recovery_heatmap.png

Figure 17. Heatmap of woody cover recovery (% of immediate loss recovered) by ecoregion at 1, 3, 5, and 10 years after a single fire event. Values of 100% indicate full recovery to pre-fire loss levels; values exceeding 100% indicate overcompensation. Colour scale is centred at 100%.

E10 – eco_summary_heatmap.png

Figure 18. Comprehensive summary of fire impact across ecoregions and fire counts. Four panels show (a) affected area in km², (b) immediate woody cover loss (percentage points), (c) 5-year recovery (% of loss), and (d) 10-year recovery (% of loss). Rows represent fire count categories; columns represent ecoregions.



In [39]:
# === SCHRITT 3d: STATISTISCHE TESTS – ECOREGION-VERGLEICHE ===

print("\n3d. STATISTISCHE TESTS – ECOREGION-VERGLEICHE")
print("-" * 70)

# Extrahiere Pixel-Level Metriken pro Ecoregion
print("\nExtrahiere Pixel-Level Metriken pro Ecoregion...")

eco_pixel_metrics = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregions"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    
    metrics = extract_pixel_metrics_lc(
        woody, burned, fire_counts, first_fire_idx, eco_mask,
        years_woody, years_burned, nodata,
        max_samples=MAX_SAMPLES
    )
    
    if metrics is not None:
        eco_pixel_metrics[eco_code] = metrics
        print(f"  ✓ {eco_code}: {metrics['n_pixels_total']:,} Pixel "
              f"(sampled: {metrics['n_pixels_sampled']:,})")

# Kruskal-Wallis Tests zwischen Ecoregions
eco_names_ordered = sorted(eco_pixel_metrics.keys())

kw_results_eco = {}

for metric in METRICS_TO_TEST:
    groups = [eco_pixel_metrics[eco][metric] for eco in eco_names_ordered]
    result = kruskal_wallis_with_effect_size(groups, eco_names_ordered, metric)
    
    if result is not None:
        kw_results_eco[metric] = result
        sig_str = "***" if result['p_value'] < 0.001 else (
            "**" if result['p_value'] < 0.01 else (
            "*" if result['p_value'] < 0.05 else "n.s."))
        print(f"\n  {METRIC_LABELS[metric]}:")
        print(f"    H = {result['H_statistic']:.1f}, p = {result['p_value']:.2e} {sig_str}")
        print(f"    η² = {result['eta_squared']:.4f} ({result['effect_size']})")

# Dunn's Post-hoc
dunn_results_eco = {}

if HAS_POSTHOC:
    print("\n  Dunn's Post-hoc Tests (Bonferroni) – Ecoregions:")
    for metric in METRICS_TO_TEST:
        if metric not in kw_results_eco or not kw_results_eco[metric]['significant']:
            continue
        
        result = kw_results_eco[metric]
        all_values = []
        all_labels = []
        for g, name in zip(result['groups'], result['group_names']):
            all_values.extend(g.tolist())
            all_labels.extend([name] * len(g))
        
        # Erstelle DataFrame statt numpy-Arrays zu übergeben
        dunn_input_df = pd.DataFrame({
            'value': all_values,
            'group': all_labels
        })
        
        dunn_df = sp.posthoc_dunn(
            a=dunn_input_df,
            val_col='value',
            group_col='group',
            p_adjust='bonferroni'
        )
        dunn_results_eco[metric] = dunn_df
        
        n_pairs = dunn_df.shape[0] * (dunn_df.shape[0] - 1) // 2
        sig_pairs = np.sum(dunn_df.values[np.triu_indices_from(dunn_df.values, k=1)] < 0.05)
        print(f"    {metric}: {sig_pairs}/{n_pairs} signifikante Paare")

# --- Boxplots: Metriken nach Ecoregion ---
print("\nErstelle Ecoregion-Boxplots...")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes_flat = axes.flatten()

for i, metric in enumerate(METRICS_TO_TEST):
    ax = axes_flat[i]
    
    box_data = []
    box_labels = []
    box_colors = []
    
    for eco_code in eco_names_ordered:
        vals = eco_pixel_metrics[eco_code][metric]
        valid = vals[~np.isnan(vals)]
        if len(valid) > 0:
            box_data.append(valid)
            box_labels.append(eco_code)
            box_colors.append(eco_results.get(eco_code, {}).get('hex_color', '#808080'))
    
    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True,
                    showfliers=False, widths=0.6)
    
    for j, color in enumerate(box_colors):
        bp['boxes'][j].set_facecolor(color)
        bp['boxes'][j].set_alpha(0.6)
    
    ax.set_title(METRIC_LABELS[metric], fontsize=11, fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    if metric in kw_results_eco:
        kw = kw_results_eco[metric]
        sig_str = "***" if kw['p_value'] < 0.001 else (
            "**" if kw['p_value'] < 0.01 else (
            "*" if kw['p_value'] < 0.05 else "n.s."))
        ax.text(0.02, 0.98, f"KW: H={kw['H_statistic']:.0f}, η²={kw['eta_squared']:.3f} {sig_str}",
                transform=ax.transAxes, fontsize=8, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

axes_flat[-1].axis('off')

plt.suptitle('Pixel-Level Metric Distributions by Ecoregion (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "statistical_tests_ecoregion_boxplots.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ statistical_tests_ecoregion_boxplots.png")

# --- Dunn Heatmaps ---
if HAS_POSTHOC and dunn_results_eco:
    n_metrics_sig = len(dunn_results_eco)
    fig, axes_dunn = plt.subplots(1, n_metrics_sig, figsize=(7 * n_metrics_sig, 6))
    if n_metrics_sig == 1:
        axes_dunn = [axes_dunn]
    
    plot_i = 0
    for metric in METRICS_TO_TEST:
        if metric not in dunn_results_eco:
            continue
        ax = axes_dunn[plot_i]
        dunn_df = dunn_results_eco[metric]
        
        log_p = -np.log10(dunn_df.values.astype(float))
        log_p = np.clip(log_p, 0, 50)
        
        annot = dunn_df.copy()
        for r_idx in annot.index:
            for c_idx in annot.columns:
                p = dunn_df.loc[r_idx, c_idx]
                if r_idx == c_idx:
                    annot.loc[r_idx, c_idx] = ""
                elif p < 0.001:
                    annot.loc[r_idx, c_idx] = "***"
                elif p < 0.01:
                    annot.loc[r_idx, c_idx] = "**"
                elif p < 0.05:
                    annot.loc[r_idx, c_idx] = "*"
                else:
                    annot.loc[r_idx, c_idx] = "n.s."
        
        sns.heatmap(
            pd.DataFrame(log_p, index=dunn_df.index, columns=dunn_df.columns),
            cmap='RdYlGn_r', annot=annot.values, fmt='',
            ax=ax, square=True, linewidths=0.5,
            cbar_kws={'label': '-log₁₀(p)'}, vmin=0, vmax=10
        )
        ax.set_title(f'{METRIC_LABELS[metric]}\n(Dunn Post-hoc, Bonferroni)',
                     fontsize=10, fontweight='bold')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
        plot_i += 1
    
    plt.suptitle("Pairwise Ecoregion Comparisons – Dunn's Post-hoc Test",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "statistical_tests_ecoregion_dunn_heatmaps.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ statistical_tests_ecoregion_dunn_heatmaps.png")

# === CSV EXPORTS ===
print("\nSpeichere Ecoregion-Statistik CSVs...")

# KW Zusammenfassung
kw_eco_rows = []
for metric in METRICS_TO_TEST:
    if metric in kw_results_eco:
        kw = kw_results_eco[metric]
        kw_eco_rows.append({
            'Metric': metric, 'Metric_Label': METRIC_LABELS[metric],
            'H_Statistic': kw['H_statistic'], 'p_value': kw['p_value'],
            'Eta_Squared': kw['eta_squared'], 'Effect_Size': kw['effect_size'],
            'k_Groups': kw['k_groups'], 'n_Total': kw['n_total'],
            'Significant_0.05': kw['significant']
        })

pd.DataFrame(kw_eco_rows).to_csv(
    os.path.join(out_dir, "statistical_tests_ecoregion_kruskal_wallis.csv"), index=False)

# Mediane pro Ecoregion
median_eco_rows = []
for eco_code in eco_names_ordered:
    row = {'Ecoregion': eco_code, 'N_Pixels_Sampled': eco_pixel_metrics[eco_code]['n_pixels_sampled']}
    for metric in METRICS_TO_TEST:
        vals = eco_pixel_metrics[eco_code][metric]
        row[f'{metric}_median'] = np.nanmedian(vals)
        row[f'{metric}_iqr'] = stats.iqr(vals[~np.isnan(vals)]) if np.sum(~np.isnan(vals)) > 0 else np.nan
    median_eco_rows.append(row)

pd.DataFrame(median_eco_rows).to_csv(
    os.path.join(out_dir, "statistical_tests_ecoregion_medians.csv"), index=False)

# Dunn Post-hoc p-Werte
if HAS_POSTHOC and dunn_results_eco:
    for metric, dunn_df in dunn_results_eco.items():
        dunn_df.to_csv(os.path.join(out_dir, f"statistical_tests_ecoregion_dunn_{metric}.csv"))

print(f"\n✓ Alle Ecoregion-Statistiken abgeschlossen!")


3d. STATISTISCHE TESTS – ECOREGION-VERGLEICHE
----------------------------------------------------------------------

Extrahiere Pixel-Level Metriken pro Ecoregion...


Ecoregions:   0%|          | 0/11 [00:00<?, ?it/s]C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:77: RuntimeWarning: divide by zero encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:77: RuntimeWarning: invalid value encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:81: RuntimeWarning: divide by zero encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100,
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\1234118159.py:81: RuntimeWarning: invalid value encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100,
Ecoregions:   9%|▉         | 1/11 [00:09<01:30,  9.08s/it]

  ✓ Anatolian: 74,792 Pixel (sampled: 50,000)


Ecoregions:  18%|█▊        | 2/11 [00:11<00:44,  4.91s/it]

  ✓ BlackSea: 8,342 Pixel (sampled: 8,342)


...

Ecoregions: 100%|██████████| 11/11 [01:19<00:00,  7.21s/it]

  ✓ Arctic: 915 Pixel (sampled: 915)

  Pre-Fire Woody Cover (Mean, %):
    H = 130747.6, p = 0.00e+00 ***
    η² = 0.3039 (large)

  Pre-Fire Trend (Slope, %/yr):
    H = 6845.3, p = 0.00e+00 ***
    η² = 0.0159 (small)



  Absolute Fire Loss (%):
    H = 1629.7, p = 0.00e+00 ***
    η² = 0.0038 (negligible)

  Relative Fire Loss (%):
    H = 1348.0, p = 1.66e-283 ***
    η² = 0.0033 (negligible)

  5-Year Recovery (% of Loss):
    H = 3075.3, p = 0.00e+00 ***
    η² = 0.0288 (small)

  Dunn's Post-hoc Tests (Bonferroni) – Ecoregions:
    pre_fire_mean: 53/55 signifikante Paare
    pre_fire_trend: 49/55 signifikante Paare
    fire_loss: 44/55 signifikante Paare
    fire_loss_pct: 42/55 signifikante Paare
    recovery_5yr_pct: 45/55 signifikante Paare

Erstelle Ecoregion-Boxplots...
  ✓ statistical_tests_ecoregion_boxplots.png


C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\209311138.py:142: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\209311138.py:142: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_131400\209311138.py:142: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))


  ✓ statistical_tests_ecoregion_dunn_heatmaps.png

Speichere Ecoregion-Statistik CSVs...

✓ Alle Ecoregion-Statistiken abgeschlossen!


Schritt 3d: Statistische Tests – Ecoregion
statistical_tests_ecoregion_boxplots.png

Figure 19. Box plot distributions of pixel-level fire impact metrics by ecoregion (single fire events). Metrics shown are: pre-fire mean woody cover, pre-fire trend, absolute and relative fire loss, and 5-year recovery. Boxes are coloured by ecoregion; outliers are suppressed. Kruskal–Wallis H-statistic, effect size (η²), and significance level are annotated per panel.

statistical_tests_ecoregion_dunn_heatmaps.png

Figure 20. Pairwise post-hoc comparisons (Dunn's test with Bonferroni correction) between ecoregions for each metric where the Kruskal–Wallis test was significant (p < 0.05). Cell colours represent −log₁₀(p); annotations indicate significance levels (*** p < 0.001, ** p < 0.01, * p < 0.05, n.s. = not significant).



In [40]:
# === SCHRITT 4: SPEICHERE ERGEBNISSE ===

print("\n4. SPEICHERE ERGEBNISSE")
print("-" * 70)

# === CSV 1: Trajektorien nach Major LC (1 Fire) ===
rows = []
for lc_class, data in lc_major_results.items():
    for i, ry in enumerate(rel_years):
        rows.append({
            'LC_Major': lc_class,
            'N_Pixels': data['n_pixels'],
            'Rel_Year': ry,
            'Woody_Cover_Mean': data['trajectory'][i],
            'Woody_Cover_Std': data['std'][i]
        })
df_traj_major = pd.DataFrame(rows)
csv1 = os.path.join(out_dir, "trajectories_by_landcover_1fire.csv")
df_traj_major.to_csv(csv1, index=False)
print(f"  ✓ {csv1}")

# === CSV 2: Trajektorien nach LC × Fire Count ===
rows = []
for lc_class, fc_data in lc_firecount_results.items():
    for fc, data in fc_data.items():
        for i, ry in enumerate(rel_years):
            rows.append({
                'LC_Major': lc_class,
                'Fire_Count': fc,
                'N_Pixels': data['n_pixels'],
                'Rel_Year': ry,
                'Woody_Cover_Mean': data['trajectory'][i],
                'Woody_Cover_Std': data['std'][i]
            })
df_traj_fc = pd.DataFrame(rows)
csv2 = os.path.join(out_dir, "trajectories_by_landcover_x_firecount.csv")
df_traj_fc.to_csv(csv2, index=False)
print(f"  ✓ {csv2}")

# === CSV 3: Trajektorien nach LC × Ecoregion ===
rows = []
for eco_code, lc_data in lc_eco_results.items():
    for lc_class, data in lc_data.items():
        for i, ry in enumerate(rel_years):
            rows.append({
                'Ecoregion': eco_code,
                'LC_Major': lc_class,
                'N_Pixels': data['n_pixels'],
                'Rel_Year': ry,
                'Woody_Cover_Mean': data['trajectory'][i],
                'Woody_Cover_Std': data['std'][i]
            })
df_traj_eco = pd.DataFrame(rows)
csv3 = os.path.join(out_dir, "trajectories_by_landcover_x_ecoregion.csv")
df_traj_eco.to_csv(csv3, index=False)
print(f"  ✓ {csv3}")

# === CSV 4: Recovery Summary ===
rows = []
for lc_class, data in lc_major_results.items():
    traj = np.array(data['trajectory'])
    pre_fire = np.nanmean(traj[:5])
    fire_year = traj[5]
    loss = pre_fire - fire_year
    
    for years_after in [1, 3, 5, 10]:
        idx = 5 + years_after
        if idx < len(traj):
            rec_val = traj[idx]
            rec_pct = ((rec_val - fire_year) / loss * 100) if loss > 0 else np.nan
        else:
            rec_val = np.nan
            rec_pct = np.nan
        
        rows.append({
            'LC_Major': lc_class,
            'N_Pixels': data['n_pixels'],
            'Pre_Fire_Cover': pre_fire,
            'Fire_Year_Cover': fire_year,
            'Woody_Loss': loss,
            'Years_After': years_after,
            'Recovery_Cover': rec_val,
            'Recovery_Percent': rec_pct
        })

df_recovery = pd.DataFrame(rows)
csv4 = os.path.join(out_dir, "recovery_summary_by_landcover.csv")
df_recovery.to_csv(csv4, index=False)
print(f"  ✓ {csv4}")


# === CSV 5: Trajektorien pro Ecoregion (1 Fire) ===
rows = []
for eco_code, data in eco_results.items():
    for i, ry in enumerate(rel_years):
        rows.append({
            'Ecoregion': eco_code, 'Ecoregion_Name': data['name'],
            'N_Pixels': data['n_pixels'], 'Rel_Year': ry,
            'Woody_Cover_Mean': data['trajectory'][i],
            'Woody_Cover_Std': data['std'][i]
        })
csv5 = os.path.join(out_dir, "trajectories_by_ecoregion_1fire.csv")
pd.DataFrame(rows).to_csv(csv5, index=False)
print(f"  ✓ {csv5}")

# === CSV 6: Trajektorien Ecoregion × Fire Count ===
rows = []
for eco_code, fc_data in eco_firecount_results.items():
    for fc, data in fc_data.items():
        for i, ry in enumerate(rel_years):
            rows.append({
                'Ecoregion': eco_code, 'Fire_Count': fc,
                'N_Pixels': data['n_pixels'], 'Rel_Year': ry,
                'Woody_Cover_Mean': data['trajectory'][i],
                'Woody_Cover_Std': data['std'][i]
            })
csv6 = os.path.join(out_dir, "trajectories_by_ecoregion_x_firecount.csv")
pd.DataFrame(rows).to_csv(csv6, index=False)
print(f"  ✓ {csv6}")

# === CSV 7: Burned Area Total ===
csv7 = os.path.join(out_dir, "burned_area_annual_total.csv")
burned_area_total.to_csv(csv7, index=False)
print(f"  ✓ {csv7}")

# === CSV 8: Burned Area pro Ecoregion ===
csv8 = os.path.join(out_dir, "burned_area_by_ecoregion.csv")
burned_area_eco.to_csv(csv8, index=False)
print(f"  ✓ {csv8}")

# === CSV 9: Fire Count Verteilung pro Ecoregion ===
csv9 = os.path.join(out_dir, "fire_count_distribution_by_ecoregion.csv")
fire_count_eco.to_csv(csv9, index=False)
print(f"  ✓ {csv9}")

# === CSV 10: Season Length Stats ===
csv10 = os.path.join(out_dir, "fire_season_length_by_ecoregion.csv")
season_stats.to_csv(csv10, index=False)
print(f"  ✓ {csv10}")

# === CSV 11: Fire Stats Summary ===
csv11 = os.path.join(out_dir, "fire_statistics_summary_per_ecoregion.csv")
fire_stats_summary.to_csv(csv11, index=False)
print(f"  ✓ {csv11}")

# === CSV 12: Recovery pro Ecoregion ===
csv12 = os.path.join(out_dir, "recovery_summary_by_ecoregion.csv")
df_eco_recovery.to_csv(csv12, index=False)
print(f"  ✓ {csv12}")

# === CSV 13: Summary Ecoregion × Fire Count ===
csv13 = os.path.join(out_dir, "summary_ecoregion_x_firecount.csv")
df_summary.to_csv(csv13, index=False)
print(f"  ✓ {csv13}")


# === ABSCHLUSSMELDUNG ===
print("\n" + "=" * 80)
print("✓✓✓ LANDCOVER- & ECOREGION-STRATIFIZIERTE ANALYSE ABGESCHLOSSEN! ✓✓✓")
print("=" * 80)

print(f"\n📊 ANALYSIERTE DATEN:")
print(f"  Landcover-Typen: {len(lc_major_results)}")
print(f"  IGBP Einzelklassen: {len(lc_detail_results)}")
print(f"  LC × Fire Count Kombinationen: {sum(len(v) for v in lc_firecount_results.values())}")
print(f"  LC × Ecoregion Kombinationen: {sum(len(v) for v in lc_eco_results.values())}")
print(f"  Ecoregionen: {len(eco_results)}")
print(f"  Ecoregion × Fire Count Kombinationen: {sum(len(v) for v in eco_firecount_results.values())}")

print(f"\n📈 PLOTS (Landcover):")
print(f"  1. lc_trajectories_1fire_all.png        – Alle LC-Typen (1 Fire)")
print(f"  2. lc_trajectories_1fire_panel.png       – Subplots pro LC-Typ")
print(f"  3. lc_trajectories_by_fire_count.png     – LC × Fire Count")
print(f"  4. lc_ecoregion_heatmap.png              – Heatmap LC × Ecoregion")
print(f"  5. lc_trajectories_<ECO>.png             – Pro Ecoregion")
print(f"  6. statistical_tests_landcover_*.png     – Stat. Tests LC")

print(f"\n📈 PLOTS (Ecoregion):")
print(f"  E1. eco_trajectories_1fire_all.png       – Alle Ecoregionen (1 Fire)")
print(f"  E2. eco_trajectories_1fire_panel.png     – Subplots pro Ecoregion")
print(f"  E3. eco_trajectories_by_fire_count.png   – Eco × Fire Count")
print(f"  E4. burned_area_annual_total.png         – Jährl. verbrannte Fläche")
print(f"  E5. burned_area_by_ecoregion.png         – Burned Area pro Ecoregion")
print(f"  E6. fire_season_length_analysis.png      – Season Length (3 Panels)")
print(f"  E7. fire_count_analysis.png              – Fire Count (3 Panels)")
print(f"  E8. fire_season_length_by_count.png      – Season Length × Fire Count")
print(f"  E9. eco_recovery_heatmap.png             – Recovery Heatmap")
print(f"  E10. eco_summary_heatmap.png             – Summary Heatmap")
print(f"  E11. statistical_tests_ecoregion_*.png   – Stat. Tests Ecoregion")

print(f"\n📋 CSV-DATEIEN (Landcover):")
print(f"   1. trajectories_by_landcover_1fire.csv")
print(f"   2. trajectories_by_landcover_x_firecount.csv")
print(f"   3. trajectories_by_landcover_x_ecoregion.csv")
print(f"   4. recovery_summary_by_landcover.csv")
print(f"   + statistical_tests_landcover_*.csv")

print(f"\n📋 CSV-DATEIEN (Ecoregion):")
print(f"   5. trajectories_by_ecoregion_1fire.csv")
print(f"   6. trajectories_by_ecoregion_x_firecount.csv")
print(f"   7. burned_area_annual_total.csv")
print(f"   8. burned_area_by_ecoregion.csv")
print(f"   9. fire_count_distribution_by_ecoregion.csv")
print(f"  10. fire_season_length_by_ecoregion.csv")
print(f"  11. fire_statistics_summary_per_ecoregion.csv")
print(f"  12. recovery_summary_by_ecoregion.csv")
print(f"  13. summary_ecoregion_x_firecount.csv")
print(f"   + statistical_tests_ecoregion_*.csv")

print(f"\n📁 Output: {out_dir}")
print("=" * 80)

# # === ABSCHLUSSMELDUNG ALT (OHNE ECOREGIONS) ===
# print("\n" + "=" * 80)
# print("✓✓✓ LANDCOVER-STRATIFIZIERTE ANALYSE ABGESCHLOSSEN! ✓✓✓")
# print("=" * 80)

# print(f"\n📊 ANALYSIERTE DATEN:")
# print(f"  Landcover-Typen: {len(lc_major_results)}")
# print(f"  IGBP Einzelklassen: {len(lc_detail_results)}")
# print(f"  LC × Fire Count Kombinationen: {sum(len(v) for v in lc_firecount_results.values())}")
# print(f"  LC × Ecoregion Kombinationen: {sum(len(v) for v in lc_eco_results.values())}")

# print(f"\n📈 PLOTS:")
# print(f"  1. lc_trajectories_1fire_all.png        – Alle LC-Typen (1 Fire)")
# print(f"  2. lc_trajectories_1fire_panel.png       – Subplots pro LC-Typ")
# print(f"  3. lc_trajectories_by_fire_count.png     – LC × Fire Count")
# print(f"  4. lc_ecoregion_heatmap.png              – Heatmap LC × Ecoregion")
# print(f"  5. lc_trajectories_<ECO>.png             – Pro Ecoregion")

# print(f"\n📋 CSV-DATEIEN:")
# print(f"  1. trajectories_by_landcover_1fire.csv")
# print(f"  2. trajectories_by_landcover_x_firecount.csv")
# print(f"  3. trajectories_by_landcover_x_ecoregion.csv")
# print(f"  4. recovery_summary_by_landcover.csv")

# print(f"\n📁 Output: {out_dir}")
# print("=" * 80)


4. SPEICHERE ERGEBNISSE
----------------------------------------------------------------------
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\trajectories_by_landcover_1fire.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\trajectories_by_landcover_x_firecount.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\trajectories_by_landcover_x_ecoregion.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\recovery_summary_by_landcover.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\trajectories_by_ecoregion_1fire.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\05_Landcover_integrated\MODIS_IGBP_Analysis\trajectories_by_ecoregion_x_firecount.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_

In [3]:
# === SCHNELLE VISUALISIERUNG AUS CSV (ohne Neuberechnung) ===
# Nutze diese Zelle, wenn du nur Plots anpassen willst

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
out_dir = os.path.join(workDir, "_Runs", "05_Landcover_integrated", "MODIS_IGBP_Analysis")

# =====================================================
# A) LANDCOVER CSVs
# =====================================================
print("Lade Landcover CSVs...")

df_traj_major = pd.read_csv(os.path.join(out_dir, "trajectories_by_landcover_1fire.csv"))
df_traj_fc = pd.read_csv(os.path.join(out_dir, "trajectories_by_landcover_x_firecount.csv"))
df_traj_eco = pd.read_csv(os.path.join(out_dir, "trajectories_by_landcover_x_ecoregion.csv"))
df_recovery_lc = pd.read_csv(os.path.join(out_dir, "recovery_summary_by_landcover.csv"))

print(f"  ✓ Major LC:     {len(df_traj_major):,} Zeilen")
print(f"  ✓ LC×FireCount: {len(df_traj_fc):,} Zeilen")
print(f"  ✓ LC×Ecoregion: {len(df_traj_eco):,} Zeilen")
print(f"  ✓ Recovery LC:  {len(df_recovery_lc):,} Zeilen")

# =====================================================
# B) ECOREGION CSVs
# =====================================================
print("\nLade Ecoregion CSVs...")

df_traj_eco_1fire = pd.read_csv(os.path.join(out_dir, "trajectories_by_ecoregion_1fire.csv"))
df_traj_eco_fc = pd.read_csv(os.path.join(out_dir, "trajectories_by_ecoregion_x_firecount.csv"))
df_burned_total = pd.read_csv(os.path.join(out_dir, "burned_area_annual_total.csv"))
df_burned_eco = pd.read_csv(os.path.join(out_dir, "burned_area_by_ecoregion.csv"))
df_fire_count_eco = pd.read_csv(os.path.join(out_dir, "fire_count_distribution_by_ecoregion.csv"))
df_season_stats = pd.read_csv(os.path.join(out_dir, "fire_season_length_by_ecoregion.csv"))
df_fire_summary = pd.read_csv(os.path.join(out_dir, "fire_statistics_summary_per_ecoregion.csv"))
df_recovery_eco = pd.read_csv(os.path.join(out_dir, "recovery_summary_by_ecoregion.csv"))
df_summary_eco_fc = pd.read_csv(os.path.join(out_dir, "summary_ecoregion_x_firecount.csv"))

print(f"  ✓ Eco 1Fire:       {len(df_traj_eco_1fire):,} Zeilen")
print(f"  ✓ Eco×FireCount:   {len(df_traj_eco_fc):,} Zeilen")
print(f"  ✓ Burned Total:    {len(df_burned_total):,} Zeilen")
print(f"  ✓ Burned Eco:      {len(df_burned_eco):,} Zeilen")
print(f"  ✓ Fire Count Eco:  {len(df_fire_count_eco):,} Zeilen")
print(f"  ✓ Season Stats:    {len(df_season_stats):,} Zeilen")
print(f"  ✓ Fire Summary:    {len(df_fire_summary):,} Zeilen")
print(f"  ✓ Recovery Eco:    {len(df_recovery_eco):,} Zeilen")
print(f"  ✓ Summary Eco×FC:  {len(df_summary_eco_fc):,} Zeilen")

# =====================================================
# C) ECOREGION-ATTRIBUTE LADEN (für Farben)
# =====================================================
print("\nLade Ecoregion-Attribute...")

ecoregion_attr_path = os.path.join(
    workDir,
    r"_Runs\04_Ecoregions_MBA\ecoregions_attributes_with_colors.csv"
)

ecoregion_df = pd.read_csv(ecoregion_attr_path)
eco_color_map = dict(zip(ecoregion_df['code'], ecoregion_df['hex_color']))
eco_name_map = dict(zip(ecoregion_df['code'], ecoregion_df['name']))

print(f"  ✓ {len(eco_color_map)} Ecoregion-Farben geladen")

# =====================================================
# D) REKONSTRUIERE DICTIONARIES
# =====================================================
print("\nRekonstruiere Dictionaries aus CSVs...")

rel_years = sorted(df_traj_major['Rel_Year'].unique())

# --- Landcover Dictionaries ---
lc_major_results = {}
for lc_class, grp in df_traj_major.groupby('LC_Major'):
    grp = grp.sort_values('Rel_Year')
    lc_major_results[lc_class] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std': grp['Woody_Cover_Std'].tolist(),
        'n_pixels': int(grp['N_Pixels'].iloc[0])
    }

lc_firecount_results = {}
for (lc_class, fc), grp in df_traj_fc.groupby(['LC_Major', 'Fire_Count']):
    grp = grp.sort_values('Rel_Year')
    if lc_class not in lc_firecount_results:
        lc_firecount_results[lc_class] = {}
    lc_firecount_results[lc_class][int(fc)] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std': grp['Woody_Cover_Std'].tolist(),
        'n_pixels': int(grp['N_Pixels'].iloc[0])
    }

lc_eco_results = {}
for (eco_code, lc_class), grp in df_traj_eco.groupby(['Ecoregion', 'LC_Major']):
    grp = grp.sort_values('Rel_Year')
    if eco_code not in lc_eco_results:
        lc_eco_results[eco_code] = {}
    lc_eco_results[eco_code][lc_class] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std': grp['Woody_Cover_Std'].tolist(),
        'n_pixels': int(grp['N_Pixels'].iloc[0])
    }

# --- Ecoregion Dictionaries ---
eco_results = {}
for (eco_code), grp in df_traj_eco_1fire.groupby('Ecoregion'):
    grp = grp.sort_values('Rel_Year')
    eco_name = grp['Ecoregion_Name'].iloc[0]
    eco_results[eco_code] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std': grp['Woody_Cover_Std'].tolist(),
        'n_pixels': int(grp['N_Pixels'].iloc[0]),
        'name': eco_name,
        'hex_color': eco_color_map.get(eco_code, '#808080')
    }

eco_firecount_results = {}
for (eco_code, fc), grp in df_traj_eco_fc.groupby(['Ecoregion', 'Fire_Count']):
    grp = grp.sort_values('Rel_Year')
    if eco_code not in eco_firecount_results:
        eco_firecount_results[eco_code] = {}
    eco_firecount_results[eco_code][int(fc)] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std': grp['Woody_Cover_Std'].tolist(),
        'n_pixels': int(grp['N_Pixels'].iloc[0])
    }

# --- Fire Stats Dictionaries ---
burned_area_total = df_burned_total
burned_area_eco = df_burned_eco
fire_count_eco = df_fire_count_eco
season_stats = df_season_stats
fire_stats_summary = df_fire_summary
df_eco_recovery = df_recovery_eco
df_summary = df_summary_eco_fc

# --- Farb-Konfigurationen ---
LC_COLORS = {
    'Forests': '#228B22',
    'Shrublands': '#8B4513',
    'Savannas': '#DAA520',
    'Grasslands': '#90EE90',
    'Wetlands': '#4682B4',
    'Croplands': '#FFD700',
    'Barren/Ice': '#D3D3D3'
}

FIRE_COUNT_COLORS = {
    1: '#1f77b4',
    2: '#ff7f0e',
    3: '#2ca02c',
    4: '#d62728'
}

# =====================================================
# ZUSAMMENFASSUNG
# =====================================================
print(f"\n✓ Alle Dictionaries rekonstruiert – Plots können direkt ausgeführt werden!")
print(f"\n  LANDCOVER:")
print(f"    {len(lc_major_results)} Major LC Klassen")
print(f"    {sum(len(v) for v in lc_firecount_results.values())} LC×FireCount Kombinationen")
print(f"    {sum(len(v) for v in lc_eco_results.values())} LC×Ecoregion Kombinationen")
print(f"\n  ECOREGION:")
print(f"    {len(eco_results)} Ecoregionen")
print(f"    {sum(len(v) for v in eco_firecount_results.values())} Eco×FireCount Kombinationen")
print(f"    {len(burned_area_total)} Jahre Burned Area (gesamt)")
print(f"    {len(burned_area_eco)} Eco×Jahr Burned Area Einträge")
print(f"    {len(fire_stats_summary)} Ecoregion Fire Statistics")

print(f"\n  → Führe Schritt 3 (LC-Plots) und 3c (Eco-Plots) direkt aus!")

Lade Landcover CSVs...
  ✓ Major LC:     112 Zeilen
  ✓ LC×FireCount: 432 Zeilen
  ✓ LC×Ecoregion: 992 Zeilen
  ✓ Recovery LC:  28 Zeilen

Lade Ecoregion CSVs...
  ✓ Eco 1Fire:       176 Zeilen
  ✓ Eco×FireCount:   656 Zeilen
  ✓ Burned Total:    26 Zeilen
  ✓ Burned Eco:      286 Zeilen
  ✓ Fire Count Eco:  153 Zeilen
  ✓ Season Stats:    273 Zeilen
  ✓ Fire Summary:    11 Zeilen
  ✓ Recovery Eco:    44 Zeilen
  ✓ Summary Eco×FC:  41 Zeilen

Lade Ecoregion-Attribute...
  ✓ 11 Ecoregion-Farben geladen

Rekonstruiere Dictionaries aus CSVs...

✓ Alle Dictionaries rekonstruiert – Plots können direkt ausgeführt werden!

  LANDCOVER:
    7 Major LC Klassen
    27 LC×FireCount Kombinationen
    62 LC×Ecoregion Kombinationen

  ECOREGION:
    11 Ecoregionen
    41 Eco×FireCount Kombinationen
    26 Jahre Burned Area (gesamt)
    286 Eco×Jahr Burned Area Einträge
    11 Ecoregion Fire Statistics

  → Führe Schritt 3 (LC-Plots) und 3c (Eco-Plots) direkt aus!
